# Baseline Comparison: PureRAG vs TitanOnly vs HybridRAG vs AdaptiveHybridRAG

This notebook compares retrieval strategies:
- **PureRAG**: BM25 + Embedding (no Memory)
- **TitanOnly**: Memory-guided retrieval only
- **HybridRAG**: BM25 + Memory + Embedding fusion
- **AdaptiveHybridRAG**: Per-Chunk Soft Routing (XGBoost scorer)

### Modes:
1. **Sample Essays Mode**: Use built-in climate/ai/space essays
2. **SQuAD Mode**: Use processed SQuAD dataset (single-doc per title)
3. **Multi-Doc Mode**: Digest multiple articles into shared memory for cross-document retrieval
4. **HotpotQA Mode**: Multi-hop QA across multiple supporting documents (already multi-document)

## 1. Install Dependencies

In [ ]:
# Install required packages
!pip install -q torch sentence-transformers transformers tqdm scikit-learn xgboost joblib

## 2. Setup File System (Core Files)

In [ ]:
import os
import base64

print(f'Current working directory: {os.getcwd()}')

def write_file(path, b64_content):
    target_path = os.path.abspath(path)
    directory = os.path.dirname(target_path)
    if directory:
        os.makedirs(directory, exist_ok=True)
    with open(target_path, 'wb') as f:
        f.write(base64.b64decode(b64_content))
    print(f'✓ {path}')

# Core files (embedded for convenience)
files = {
    'src/main.py': '...',
    'data/sample_essays.py': 'IiIiClNhbXBsZSBFc3NheXMgZm9yIFRpdGFuUkFHIEV4cGVyaW1lbnRzCgpUaGVzZSBlc3NheXMgY29udGFpbiBmYWN0dWFsIGluZm9ybWF0aW9uIHRoYXQgY2FuIGJlIHVzZWQgdG8gdGVzdAp0aGUgbW9kZWwncyBhYmlsaXR5IHRvIG1lbW9yaXplIGFuZCByZWNhbGwgc3BlY2lmaWMgZGV0YWlscy4KIiIiCgpFU1NBWV9DTElNQVRFID0gIiIiCiMgVGhlIFNjaWVuY2Ugb2YgQ2xpbWF0ZSBDaGFuZ2UKCiMjIEludHJvZHVjdGlvbgpDbGltYXRlIGNoYW5nZSByZWZlcnMgdG8gbG9uZy10ZXJtIHNoaWZ0cyBpbiBnbG9iYWwgdGVtcGVyYXR1cmVzIGFuZCB3ZWF0aGVyIHBhdHRlcm5zLiAKV2hpbGUgbmF0dXJhbCBmYWN0b3JzIGxpa2Ugdm9sY2FuaWMgZXJ1cHRpb25zIGFuZCBzb2xhciBjeWNsZXMgY2FuIGluZmx1ZW5jZSBjbGltYXRlLCAKaHVtYW4gYWN0aXZpdGllcyBoYXZlIGJlZW4gdGhlIHByaW1hcnkgZHJpdmVyIHNpbmNlIHRoZSAxODAwcy4KCiMjIEtleSBGYWN0cwotIFRoZSBnbG9iYWwgYXZlcmFnZSB0ZW1wZXJhdHVyZSBoYXMgcmlzZW4gYnkgYXBwcm94aW1hdGVseSAxLjHCsEMgc2luY2UgcHJlLWluZHVzdHJpYWwgdGltZXMuCi0gVGhlIGNvbmNlbnRyYXRpb24gb2YgQ08yIGluIHRoZSBhdG1vc3BoZXJlIHJlYWNoZWQgNDIxIHBhcnRzIHBlciBtaWxsaW9uIChwcG0pIGluIDIwMjMuCi0gU2VhIGxldmVscyBoYXZlIHJpc2VuIGJ5IGFib3V0IDIwIGNlbnRpbWV0ZXJzICg4IGluY2hlcykgc2luY2UgMTkwMC4KLSBUaGUgQXJjdGljIGlzIHdhcm1pbmcgbmVhcmx5IGZvdXIgdGltZXMgZmFzdGVyIHRoYW4gdGhlIGdsb2JhbCBhdmVyYWdlLgoKIyMgQ2F1c2VzClRoZSBidXJuaW5nIG9mIGZvc3NpbCBmdWVsc+KAlGNvYWwsIG9pbCwgYW5kIG5hdHVyYWwgZ2Fz4oCUcmVsZWFzZXMgY2FyYm9uIGRpb3hpZGUgKENPMikgCmFuZCBvdGhlciBncmVlbmhvdXNlIGdhc2VzIGludG8gdGhlIGF0bW9zcGhlcmUuIFRoZXNlIGdhc2VzIHRyYXAgaGVhdCBmcm9tIHRoZSBzdW4sIApjcmVhdGluZyBhICJncmVlbmhvdXNlIGVmZmVjdCIgdGhhdCB3YXJtcyB0aGUgcGxhbmV0LgoKIyMgSW1wYWN0cwoxLiAqKkV4dHJlbWUgV2VhdGhlcioqOiBNb3JlIGZyZXF1ZW50IGhlYXR3YXZlcywgZHJvdWdodHMsIGFuZCBpbnRlbnNlIHN0b3Jtcy4KMi4gKipFY29zeXN0ZW0gRGlzcnVwdGlvbioqOiBDb3JhbCBibGVhY2hpbmcsIHNwZWNpZXMgbWlncmF0aW9uLCBhbmQgYmlvZGl2ZXJzaXR5IGxvc3MuCjMuICoqSHVtYW4gSGVhbHRoKio6IEluY3JlYXNlZCBoZWF0LXJlbGF0ZWQgaWxsbmVzc2VzIGFuZCBzcHJlYWQgb2YgdmVjdG9yLWJvcm5lIGRpc2Vhc2VzLgo0LiAqKkFncmljdWx0dXJlKio6IFNoaWZ0cyBpbiBncm93aW5nIHNlYXNvbnMgYW5kIHJlZHVjZWQgY3JvcCB5aWVsZHMgaW4gc29tZSByZWdpb25zLgoKIyMgU29sdXRpb25zCi0gVHJhbnNpdGlvbiB0byByZW5ld2FibGUgZW5lcmd5IHNvdXJjZXMgKHNvbGFyLCB3aW5kLCBoeWRybykuCi0gSW1wcm92ZSBlbmVyZ3kgZWZmaWNpZW5jeSBpbiBidWlsZGluZ3MgYW5kIHRyYW5zcG9ydGF0aW9uLgotIFByb3RlY3QgYW5kIHJlc3RvcmUgZm9yZXN0cywgd2hpY2ggYWJzb3JiIENPMi4KLSBJbXBsZW1lbnQgY2FyYm9uIHByaWNpbmcgdG8gaW5jZW50aXZpemUgZW1pc3Npb24gcmVkdWN0aW9ucy4KCiMjIENvbmNsdXNpb24KQWRkcmVzc2luZyBjbGltYXRlIGNoYW5nZSByZXF1aXJlcyB1cmdlbnQsIGNvb3JkaW5hdGVkIGdsb2JhbCBhY3Rpb24uIFRoZSBQYXJpcyBBZ3JlZW1lbnQgCm9mIDIwMTUgc2V0IHRoZSBnb2FsIG9mIGxpbWl0aW5nIHdhcm1pbmcgdG8gMS41wrBDIGFib3ZlIHByZS1pbmR1c3RyaWFsIGxldmVscy4KIiIiCgpFU1NBWV9BSSA9ICIiIgojIFRoZSBFdm9sdXRpb24gb2YgQXJ0aWZpY2lhbCBJbnRlbGxpZ2VuY2UKCiMjIEhpc3RvcmljYWwgQmFja2dyb3VuZApUaGUgdGVybSAiQXJ0aWZpY2lhbCBJbnRlbGxpZ2VuY2UiIHdhcyBjb2luZWQgYnkgSm9obiBNY0NhcnRoeSBpbiAxOTU2IGF0IHRoZSBEYXJ0bW91dGggCkNvbmZlcmVuY2UuIEVhcmx5IEFJIHJlc2VhcmNoIGZvY3VzZWQgb24gc3ltYm9saWMgcmVhc29uaW5nIGFuZCBwcm9ibGVtLXNvbHZpbmcuCgojIyBUaW1lbGluZSBvZiBNYWpvciBNaWxlc3RvbmVzCi0gKioxOTUwKio6IEFsYW4gVHVyaW5nIHByb3Bvc2VzIHRoZSAiVHVyaW5nIFRlc3QiIGZvciBtYWNoaW5lIGludGVsbGlnZW5jZS4KLSAqKjE5NTYqKjogRGFydG1vdXRoIENvbmZlcmVuY2UgbWFya3MgdGhlIGJpcnRoIG9mIEFJIGFzIGEgZmllbGQuCi0gKioxOTk3Kio6IElCTSdzIERlZXAgQmx1ZSBkZWZlYXRzIHdvcmxkIGNoZXNzIGNoYW1waW9uIEdhcnJ5IEthc3Bhcm92LgotICoqMjAxMSoqOiBJQk0gV2F0c29uIHdpbnMgSmVvcGFyZHkhIGFnYWluc3QgaHVtYW4gY2hhbXBpb25zLgotICoqMjAxNioqOiBHb29nbGUncyBBbHBoYUdvIGRlZmVhdHMgTGVlIFNlZG9sIGluIHRoZSBnYW1lIG9mIEdvLgotICoqMjAyMioqOiBDaGF0R1BUIGxhdW5jaGVzLCBkZW1vbnN0cmF0aW5nIGFkdmFuY2VkIGNvbnZlcnNhdGlvbmFsIEFJLgoKIyMgVHlwZXMgb2YgQUkKMS4gKipOYXJyb3cgQUkgKEFOSSkqKjogRGVzaWduZWQgZm9yIHNwZWNpZmljIHRhc2tzIChlLmcuLCBpbWFnZSByZWNvZ25pdGlvbiwgbGFuZ3VhZ2UgdHJhbnNsYXRpb24pLgoyLiAqKkdlbmVyYWwgQUkgKEFHSSkqKjogSHlwb3RoZXRpY2FsIEFJIHdpdGggaHVtYW4tbGV2ZWwgY29nbml0aXZlIGFiaWxpdGllcyBhY3Jvc3MgYWxsIGRvbWFpbnMuCjMuICoqU3VwZXJpbnRlbGxpZ2VudCBBSSAoQVNJKSoqOiBUaGVvcmV0aWNhbCBBSSBzdXJwYXNzaW5nIGh1bWFuIGludGVsbGlnZW5jZSBlbnRpcmVseS4KCiMjIEN1cnJlbnQgQXBwbGljYXRpb25zCi0gKipIZWFsdGhjYXJlKio6IE1lZGljYWwgaW1hZ2luZyBhbmFseXNpcywgZHJ1ZyBkaXNjb3ZlcnksIHBlcnNvbmFsaXplZCB0cmVhdG1lbnQgcGxhbnMuCi0gKipGaW5hbmNlKio6IEZyYXVkIGRldGVjdGlvbiwgYWxnb3JpdGhtaWMgdHJhZGluZywgY3JlZGl0IHNjb3JpbmcuCi0gKipUcmFuc3BvcnRhdGlvbioqOiBBdXRvbm9tb3VzIHZlaGljbGVzLCB0cmFmZmljIG9wdGltaXphdGlvbiwgcm91dGUgcGxhbm5pbmcuCi0gKipFZHVjYXRpb24qKjogUGVyc29uYWxpemVkIGxlYXJuaW5nLCBhdXRvbWF0ZWQgZ3JhZGluZywgdHV0b3Jpbmcgc3lzdGVtcy4KCiMjIEtleSBUZWNobm9sb2dpZXMKVGhlIG1vZGVybiBBSSByZXZvbHV0aW9uIGlzIGJ1aWx0IG9uOgotICoqRGVlcCBMZWFybmluZyoqOiBOZXVyYWwgbmV0d29ya3Mgd2l0aCBtdWx0aXBsZSBsYXllcnMgZm9yIGZlYXR1cmUgZXh0cmFjdGlvbi4KLSAqKlRyYW5zZm9ybWVycyoqOiBBcmNoaXRlY3R1cmUgYmVoaW5kIGxhcmdlIGxhbmd1YWdlIG1vZGVscyBsaWtlIEdQVC00IGFuZCBDbGF1ZGUuCi0gKipSZWluZm9yY2VtZW50IExlYXJuaW5nKio6IFRyYWluaW5nIGFnZW50cyB0aHJvdWdoIHJld2FyZC1iYXNlZCBmZWVkYmFjay4KCiMjIEV0aGljYWwgQ29uc2lkZXJhdGlvbnMKLSBCaWFzIGluIHRyYWluaW5nIGRhdGEgbGVhZGluZyB0byB1bmZhaXIgb3V0Y29tZXMuCi0gSm9iIGRpc3BsYWNlbWVudCBhbmQgZWNvbm9taWMgaW5lcXVhbGl0eS4KLSBQcml2YWN5IGNvbmNlcm5zIGZyb20gZGF0YSBjb2xsZWN0aW9uLgotIFBvdGVudGlhbCBtaXN1c2UgZm9yIG1pc2luZm9ybWF0aW9uIGFuZCBzdXJ2ZWlsbGFuY2UuCgojIyBUaGUgRnV0dXJlCkV4cGVydHMgcHJlZGljdCBBSSB3aWxsIGNvbnRpbnVlIHRvIHRyYW5zZm9ybSBpbmR1c3RyaWVzLCB3aXRoIGVzdGltYXRlZCBlY29ub21pYyBpbXBhY3QgCm9mICQxNS43IHRyaWxsaW9uIGJ5IDIwMzAgYWNjb3JkaW5nIHRvIFB3Qy4gVGhlIGRldmVsb3BtZW50IG9mIEFHSSByZW1haW5zIGEgbG9uZy10ZXJtIGdvYWwuCiIiIgoKRVNTQVlfU1BBQ0UgPSAiIiIKIyBFeHBsb3JpbmcgdGhlIFNvbGFyIFN5c3RlbQoKIyMgT3VyIENvc21pYyBOZWlnaGJvcmhvb2QKVGhlIFNvbGFyIFN5c3RlbSBjb25zaXN0cyBvZiB0aGUgU3VuLCBlaWdodCBwbGFuZXRzLCBkd2FyZiBwbGFuZXRzLCBtb29ucywgYXN0ZXJvaWRzLCAKYW5kIGNvbWV0cy4gSXQgZm9ybWVkIGFwcHJveGltYXRlbHkgNC42IGJpbGxpb24geWVhcnMgYWdvIGZyb20gYSBnaWFudCBtb2xlY3VsYXIgY2xvdWQuCgojIyBUaGUgU3VuCi0gKipUeXBlKio6IEctdHlwZSBtYWluLXNlcXVlbmNlIHN0YXIgKHllbGxvdyBkd2FyZikKLSAqKkRpYW1ldGVyKio6IDEsMzkyLDcwMCBrbSAoMTA5IHRpbWVzIEVhcnRoJ3MgZGlhbWV0ZXIpCi0gKipTdXJmYWNlIFRlbXBlcmF0dXJlKio6IDUsNTAwwrBDICg5LDkwMMKwRikKLSAqKkNvcmUgVGVtcGVyYXR1cmUqKjogMTUgbWlsbGlvbsKwQwotICoqRGlzdGFuY2UgZnJvbSBFYXJ0aCoqOiAxNDkuNiBtaWxsaW9uIGttICgxIEFzdHJvbm9taWNhbCBVbml0KQoKIyMgVGhlIFBsYW5ldHMKIyMjIElubmVyIFBsYW5ldHMgKFJvY2t5L1RlcnJlc3RyaWFsKQoxLiAqKk1lcmN1cnkqKjogU21hbGxlc3QgcGxhbmV0LCA4OC1kYXkgb3JiaXQsIG5vIG1vb25zLgoyLiAqKlZlbnVzKio6IEhvdHRlc3QgcGxhbmV0ICg0NzXCsEMgc3VyZmFjZSksIHJvdGF0ZXMgYmFja3dhcmRzLgozLiAqKkVhcnRoKio6IE9ubHkga25vd24gcGxhbmV0IHdpdGggbGlxdWlkIHdhdGVyIG9uIHN1cmZhY2UgYW5kIGxpZmUuCjQuICoqTWFycyoqOiAiUmVkIFBsYW5ldCIsIGhhcyB0aGUgbGFyZ2VzdCB2b2xjYW5vIChPbHltcHVzIE1vbnMsIDIxLjkga20gaGlnaCkuCgojIyMgT3V0ZXIgUGxhbmV0cyAoR2FzL0ljZSBHaWFudHMpCjUuICoqSnVwaXRlcioqOiBMYXJnZXN0IHBsYW5ldCAoMTF4IEVhcnRoJ3MgZGlhbWV0ZXIpLCBHcmVhdCBSZWQgU3BvdCBzdG9ybS4KNi4gKipTYXR1cm4qKjogRmFtb3VzIGZvciBpdHMgcmluZyBzeXN0ZW0sIDgzIGtub3duIG1vb25zIGluY2x1ZGluZyBUaXRhbi4KNy4gKipVcmFudXMqKjogUm90YXRlcyBvbiBpdHMgc2lkZSAoOTjCsCBheGlhbCB0aWx0KSwgMjcgbW9vbnMuCjguICoqTmVwdHVuZSoqOiBTdHJvbmdlc3Qgd2luZHMgaW4gdGhlIFNvbGFyIFN5c3RlbSAoMiwxMDAga20vaCkuCgojIyBOb3RhYmxlIE1pc3Npb25zCi0gKipWb3lhZ2VyIDEgJiAyICgxOTc3KSoqOiBGaXJzdCBzcGFjZWNyYWZ0IHRvIHJlYWNoIGludGVyc3RlbGxhciBzcGFjZS4KLSAqKkh1YmJsZSBTcGFjZSBUZWxlc2NvcGUgKDE5OTApKio6IFJldm9sdXRpb25pemVkIG91ciB2aWV3IG9mIHRoZSB1bml2ZXJzZS4KLSAqKk1hcnMgUm92ZXJzKio6IFNwaXJpdCwgT3Bwb3J0dW5pdHksIEN1cmlvc2l0eSwgUGVyc2V2ZXJhbmNlIGV4cGxvcmVkIE1hcnMuCi0gKipKYW1lcyBXZWJiIFNwYWNlIFRlbGVzY29wZSAoMjAyMSkqKjogTW9zdCBwb3dlcmZ1bCBzcGFjZSB0ZWxlc2NvcGUgZXZlciBidWlsdC4KCiMjIER3YXJmIFBsYW5ldHMKUGx1dG8gd2FzIHJlY2xhc3NpZmllZCBhcyBhIGR3YXJmIHBsYW5ldCBpbiAyMDA2IGJ5IHRoZSBJbnRlcm5hdGlvbmFsIEFzdHJvbm9taWNhbCBVbmlvbi4gCk90aGVyIGR3YXJmIHBsYW5ldHMgaW5jbHVkZSBFcmlzLCBIYXVtZWEsIE1ha2VtYWtlLCBhbmQgQ2VyZXMuCgojIyBGdXR1cmUgb2YgU3BhY2UgRXhwbG9yYXRpb24KLSAqKjIwMjQtMjAyNSoqOiBBcnRlbWlzIHByb2dyYW0gYWltcyB0byByZXR1cm4gaHVtYW5zIHRvIHRoZSBNb29uLgotICoqMjAzMHMqKjogTkFTQSBhbmQgU3BhY2VYIHBsYW5uaW5nIGNyZXdlZCBtaXNzaW9ucyB0byBNYXJzLgotICoqTG9uZy10ZXJtKio6IFBvdGVudGlhbCBjb2xvbml6YXRpb24gb2YgTWFycyBhbmQgYXN0ZXJvaWQgbWluaW5nLgoiIiIKCiMgUXVlc3Rpb25zIGZvciB0ZXN0aW5nIC0gTWl4ZWQgZmFjdHVhbCBhbmQgcmVhc29uaW5nIHF1ZXN0aW9ucwojIEZhY3R1YWwgcXVlc3Rpb25zICh3aGF0L3doZW4vd2hlcmUpIHNob3VsZCB0cmlnZ2VyIEhZQlJJRCBtb2RlCiMgUmVhc29uaW5nIHF1ZXN0aW9ucyAod2h5L2V4cGxhaW4vZGVzY3JpYmUpIHNob3VsZCB0cmlnZ2VyIFRJVEFOU19PTkxZIG1vZGUKVEVTVF9RVUVTVElPTlMgPSB7CiAgICAiY2xpbWF0ZSI6IFsKICAgICAgICAjIEZhY3R1YWwgcXVlc3Rpb25zIChleHBlY3QgSFlCUklEKQogICAgICAgICgiV2hhdCBpcyB0aGUgY3VycmVudCBDTzIgY29uY2VudHJhdGlvbiBpbiB0aGUgYXRtb3NwaGVyZT8iLCAiNDIxIHBhcnRzIHBlciBtaWxsaW9uIChwcG0pIGluIDIwMjMiKSwKICAgICAgICAoIkhvdyBtdWNoIGhhcyB0aGUgZ2xvYmFsIHRlbXBlcmF0dXJlIHJpc2VuIHNpbmNlIHByZS1pbmR1c3RyaWFsIHRpbWVzPyIsICJhcHByb3hpbWF0ZWx5IDEuMcKwQyIpLAogICAgICAgICgiV2hhdCB5ZWFyIHdhcyB0aGUgUGFyaXMgQWdyZWVtZW50IHNpZ25lZD8iLCAiMjAxNSIpLAogICAgICAgICgiSG93IG11Y2ggaGF2ZSBzZWEgbGV2ZWxzIHJpc2VuIHNpbmNlIDE5MDA/IiwgImFib3V0IDIwIGNlbnRpbWV0ZXJzICg4IGluY2hlcykiKSwKICAgICAgICAKICAgICAgICAjIFJlYXNvbmluZyBxdWVzdGlvbnMgKGV4cGVjdCBUSVRBTlNfT05MWSkKICAgICAgICAoIldoeSBpcyBjbGltYXRlIGNoYW5nZSBwcmltYXJpbHkgY2F1c2VkIGJ5IGh1bWFuIGFjdGl2aXRpZXM/IiwgImJ1cm5pbmcgb2YgZm9zc2lsIGZ1ZWxzIHJlbGVhc2VzIGdyZWVuaG91c2UgZ2FzZXMgdGhhdCB0cmFwIGhlYXQiKSwKICAgICAgICAoIkV4cGxhaW4gaG93IHRoZSBncmVlbmhvdXNlIGVmZmVjdCB3b3JrcyIsICJncmVlbmhvdXNlIGdhc2VzIHRyYXAgaGVhdCBmcm9tIHRoZSBzdW4sIGNyZWF0aW5nIHdhcm1pbmcgZWZmZWN0IiksCiAgICAgICAgKCJEZXNjcmliZSB0aGUgbWFpbiBpbXBhY3RzIG9mIGNsaW1hdGUgY2hhbmdlIG9uIGVjb3N5c3RlbXMiLCAiY29yYWwgYmxlYWNoaW5nLCBzcGVjaWVzIG1pZ3JhdGlvbiwgYmlvZGl2ZXJzaXR5IGxvc3MsIGFuZCBoYWJpdGF0IGRpc3J1cHRpb24iKSwKICAgIF0sCiAgICAiYWkiOiBbCiAgICAgICAgIyBGYWN0dWFsIHF1ZXN0aW9ucyAoZXhwZWN0IEhZQlJJRCkKICAgICAgICAoIldobyBjb2luZWQgdGhlIHRlcm0gJ0FydGlmaWNpYWwgSW50ZWxsaWdlbmNlJz8iLCAiSm9obiBNY0NhcnRoeSBpbiAxOTU2IiksCiAgICAgICAgKCJXaGVuIGRpZCBEZWVwIEJsdWUgZGVmZWF0IEdhcnJ5IEthc3Bhcm92PyIsICIxOTk3IiksCiAgICAgICAgKCJXaGF0IGlzIHRoZSBlc3RpbWF0ZWQgZWNvbm9taWMgaW1wYWN0IG9mIEFJIGJ5IDIwMzA/IiwgIiQxNS43IHRyaWxsaW9uIGFjY29yZGluZyB0byBQd0MiKSwKICAgICAgICAoIldoYXQgYXJjaGl0ZWN0dXJlIHBvd2VycyBHUFQtNCBhbmQgQ2xhdWRlPyIsICJUcmFuc2Zvcm1lcnMiKSwKICAgICAgICAKICAgICAgICAjIFJlYXNvbmluZyBxdWVzdGlvbnMgKGV4cGVjdCBUSVRBTlNfT05MWSkKICAgICAgICAoIldoeSBpcyBiaWFzIGluIEFJIGEgbWFqb3IgZXRoaWNhbCBjb25jZXJuPyIsICJiaWFzIGluIHRyYWluaW5nIGRhdGEgY2FuIGxlYWQgdG8gdW5mYWlyIG91dGNvbWVzIGFuZCBkaXNjcmltaW5hdGlvbiIpLAogICAgICAgICgiRXhwbGFpbiB0aGUgZGlmZmVyZW5jZSBiZXR3ZWVuIE5hcnJvdyBBSSBhbmQgR2VuZXJhbCBBSSIsICJOYXJyb3cgQUkgaXMgZGVzaWduZWQgZm9yIHNwZWNpZmljIHRhc2tzIHdoaWxlIEdlbmVyYWwgQUkgaGFzIGh1bWFuLWxldmVsIGNvZ25pdGl2ZSBhYmlsaXRpZXMgYWNyb3NzIGFsbCBkb21haW5zIiksCiAgICAgICAgKCJEZXNjcmliZSBob3cgcmVpbmZvcmNlbWVudCBsZWFybmluZyB3b3JrcyIsICJ0cmFpbmluZyBhZ2VudHMgdGhyb3VnaCByZXdhcmQtYmFzZWQgZmVlZGJhY2sgdG8gbGVhcm4gb3B0aW1hbCBiZWhhdmlvcnMiKSwKICAgIF0sCiAgICAic3BhY2UiOiBbCiAgICAgICAgIyBGYWN0dWFsIHF1ZXN0aW9ucyAoZXhwZWN0IEhZQlJJRCkKICAgICAgICAoIkhvdyBvbGQgaXMgdGhlIFNvbGFyIFN5c3RlbT8iLCAiYXBwcm94aW1hdGVseSA0LjYgYmlsbGlvbiB5ZWFycyIpLAogICAgICAgICgiV2hhdCBpcyB0aGUgdGFsbGVzdCB2b2xjYW5vIGluIHRoZSBTb2xhciBTeXN0ZW0/IiwgIk9seW1wdXMgTW9ucywgMjEuOSBrbSBoaWdoIiksCiAgICAgICAgKCJXaGVuIHdhcyB0aGUgSmFtZXMgV2ViYiBTcGFjZSBUZWxlc2NvcGUgbGF1bmNoZWQ/IiwgIjIwMjEiKSwKICAgICAgICAoIldoYXQgeWVhciB3YXMgUGx1dG8gcmVjbGFzc2lmaWVkIGFzIGEgZHdhcmYgcGxhbmV0PyIsICIyMDA2IiksCiAgICAgICAgCiAgICAgICAgIyBSZWFzb25pbmcgcXVlc3Rpb25zIChleHBlY3QgVElUQU5TX09OTFkpCiAgICAgICAgKCJXaHkgaXMgVmVudXMgdGhlIGhvdHRlc3QgcGxhbmV0IGRlc3BpdGUgTWVyY3VyeSBiZWluZyBjbG9zZXIgdG8gdGhlIFN1bj8iLCAiVmVudXMgaGFzIGEgdGhpY2sgYXRtb3NwaGVyZSB0aGF0IHRyYXBzIGhlYXQgdGhyb3VnaCBncmVlbmhvdXNlIGVmZmVjdCIpLAogICAgICAgICgiRXhwbGFpbiB3aHkgb3V0ZXIgcGxhbmV0cyBhcmUgY2FsbGVkIGdhcyBnaWFudHMiLCAidGhleSBhcmUgbXVjaCBsYXJnZXIgYW5kIGNvbXBvc2VkIHByaW1hcmlseSBvZiBnYXNlcyBsaWtlIGh5ZHJvZ2VuIGFuZCBoZWxpdW0gcmF0aGVyIHRoYW4gc29saWQgcm9jayIpLAogICAgICAgICgiRGVzY3JpYmUgdGhlIHNpZ25pZmljYW5jZSBvZiB0aGUgVm95YWdlciBtaXNzaW9ucyIsICJmaXJzdCBzcGFjZWNyYWZ0IHRvIHJlYWNoIGludGVyc3RlbGxhciBzcGFjZSBhbmQgZXhwbG9yZSB0aGUgb3V0ZXIgc29sYXIgc3lzdGVtIiksCiAgICBdLAp9CgpkZWYgZ2V0X2FsbF9lc3NheXMoKToKICAgICIiIlJldHVybiBhbGwgZXNzYXlzIGFzIGEgZGljdGlvbmFyeS4iIiIKICAgIHJldHVybiB7CiAgICAgICAgImNsaW1hdGUiOiBFU1NBWV9DTElNQVRFLAogICAgICAgICJhaSI6IEVTU0FZX0FJLAogICAgICAgICJzcGFjZSI6IEVTU0FZX1NQQUNFLAogICAgfQoKZGVmIGdldF90ZXN0X3F1ZXN0aW9ucygpOgogICAgIiIiUmV0dXJuIHRlc3QgcXVlc3Rpb25zIHdpdGggZXhwZWN0ZWQgYW5zd2Vycy4iIiIKICAgIHJldHVybiBURVNUX1FVRVNUSU9OUwo=',
    'projects/hybrid_titans/common/__init__.py': 'IyBDb21tb24gbW9kdWxlcw==',
    'projects/hybrid_titans/common/embedders.py': 'IiIiClNlbWFudGljIEVtYmVkZGVyIHVzaW5nIHNlbnRlbmNlLXRyYW5zZm9ybWVycy4KU2hhcmVkIGJ5IGFsbCBkZW1vIHNjcmlwdHMuCiIiIgoKaW1wb3J0IHRvcmNoCmltcG9ydCB0b3JjaC5ubiBhcyBubgoKCiMgRGV0ZWN0IGRldmljZQojIERldGVjdCBkZXZpY2UKREVWSUNFID0gdG9yY2guZGV2aWNlKCJjcHUiKQppZiB0b3JjaC5jdWRhLmlzX2F2YWlsYWJsZSgpOgogICAgREVWSUNFID0gdG9yY2guZGV2aWNlKCJjdWRhIikKZWxzZToKICAgIHRyeToKICAgICAgICBpbXBvcnQgdG9yY2hfeGxhLmNvcmUueGxhX21vZGVsIGFzIHhtCiAgICAgICAgREVWSUNFID0geG0ueGxhX2RldmljZSgpCiAgICAgICAgcHJpbnQoZiLinIUgVFBVIGRldGVjdGVkOiB1c2luZyBYTEEgZGV2aWNlIHtERVZJQ0V9IikKICAgIGV4Y2VwdCBJbXBvcnRFcnJvcjoKICAgICAgICBwYXNzCgoKY2xhc3MgU2VudGVuY2VUcmFuc2Zvcm1lckVtYmVkZGVyKG5uLk1vZHVsZSk6CiAgICAiIiJTZW1hbnRpYyBlbWJlZGRlciB1c2luZyBzZW50ZW5jZS10cmFuc2Zvcm1lcnMuIiIiCiAgICAKICAgIGRlZiBfX2luaXRfXyhzZWxmLCBtb2RlbF9uYW1lPSJhbGwtTWluaUxNLUw2LXYyIiwgdGFyZ2V0X2RpbT0zODQsIGRldmljZT1Ob25lKToKICAgICAgICBzdXBlcigpLl9faW5pdF9fKCkKICAgICAgICBmcm9tIHNlbnRlbmNlX3RyYW5zZm9ybWVycyBpbXBvcnQgU2VudGVuY2VUcmFuc2Zvcm1lcgogICAgICAgIAogICAgICAgIGlmIGRldmljZSBpcyBOb25lOgogICAgICAgICAgICBkZXZpY2UgPSBERVZJQ0UKICAgICAgICBzZWxmLmRldmljZSA9IGRldmljZQogICAgICAgIAogICAgICAgICMgRm9yY2UgbW9kZWwgdG8gQ1BVIHRvIGF2b2lkIFhMQS9UUFUgaW5mZXJlbmNlIGlzc3VlcwogICAgICAgICMgKFJ1bnRpbWVFcnJvcjogQ2Fubm90IHNldCB2ZXJzaW9uX2NvdW50ZXIgZm9yIGluZmVyZW5jZSB0ZW5zb3IpCiAgICAgICAgc2VsZi5jcHVfZGV2aWNlID0gImNwdSIKICAgICAgICBzZWxmLm1vZGVsID0gU2VudGVuY2VUcmFuc2Zvcm1lcihtb2RlbF9uYW1lLCBkZXZpY2U9c2VsZi5jcHVfZGV2aWNlKQogICAgICAgIHNlbGYuc291cmNlX2RpbSA9IHNlbGYubW9kZWwuZ2V0X3NlbnRlbmNlX2VtYmVkZGluZ19kaW1lbnNpb24oKQogICAgICAgIHNlbGYudGFyZ2V0X2RpbSA9IHRhcmdldF9kaW0KICAgICAgICAKICAgICAgICBpZiBzZWxmLnNvdXJjZV9kaW0gIT0gdGFyZ2V0X2RpbToKICAgICAgICAgICAgc2VsZi5wcm9qZWN0aW9uID0gbm4uTGluZWFyKHNlbGYuc291cmNlX2RpbSwgdGFyZ2V0X2RpbSkudG8oZGV2aWNlKQogICAgICAgIGVsc2U6CiAgICAgICAgICAgIHNlbGYucHJvamVjdGlvbiA9IE5vbmUKICAgICAgICAKICAgICAgICBpZiBzdHIoZGV2aWNlKS5zdGFydHN3aXRoKCd4bGEnKToKICAgICAgICAgICAgZGV2aWNlX25hbWUgPSAiVFBVIChFbWJlZGRlciBvbiBDUFUpIgogICAgICAgIGVsaWYgZGV2aWNlLnR5cGUgPT0gImN1ZGEiOgogICAgICAgICAgICBkZXZpY2VfbmFtZSA9ICJHUFUiCiAgICAgICAgZWxzZToKICAgICAgICAgICAgZGV2aWNlX25hbWUgPSAiQ1BVIgogICAgICAgIHByaW50KGYi4pyFIExvYWRlZCBTZW50ZW5jZVRyYW5zZm9ybWVyOiB7bW9kZWxfbmFtZX0gKGRpbT17c2VsZi5zb3VyY2VfZGltfSkgb24ge2RldmljZV9uYW1lfSIpCiAgICAKICAgIGRlZiBmb3J3YXJkKHNlbGYsIHRleHQpOgogICAgICAgICIiIkVtYmVkIHRleHQgaW50byBhIGZpeGVkLXNpemUgdmVjdG9yLiIiIgogICAgICAgICMgRW1iZWQgb24gQ1BVIHRoZW4gbW92ZSB0byB0YXJnZXQgZGV2aWNlIChUUFUvR1BVKQogICAgICAgIHdpdGggdG9yY2gubm9fZ3JhZCgpOgogICAgICAgICAgICBlbWJlZGRpbmcgPSBzZWxmLm1vZGVsLmVuY29kZSh0ZXh0LCBjb252ZXJ0X3RvX3RlbnNvcj1UcnVlKQogICAgICAgICAgICBlbWJlZGRpbmcgPSBlbWJlZGRpbmcuY2xvbmUoKS50byhzZWxmLmRldmljZSkKICAgICAgICAKICAgICAgICBpZiBzZWxmLnByb2plY3Rpb24gaXMgbm90IE5vbmU6CiAgICAgICAgICAgIGVtYmVkZGluZyA9IHNlbGYucHJvamVjdGlvbihlbWJlZGRpbmcuZmxvYXQoKSkKICAgICAgICAKICAgICAgICAjIEVuc3VyZSAzRCBvdXRwdXQ6IFsxLCBzZXFfbGVuLCBkaW1dCiAgICAgICAgaWYgZW1iZWRkaW5nLmRpbSgpID09IDE6CiAgICAgICAgICAgIGVtYmVkZGluZyA9IGVtYmVkZGluZy51bnNxdWVlemUoMCkudW5zcXVlZXplKDApCiAgICAgICAgZWxpZiBlbWJlZGRpbmcuZGltKCkgPT0gMjoKICAgICAgICAgICAgZW1iZWRkaW5nID0gZW1iZWRkaW5nLnVuc3F1ZWV6ZSgwKQogICAgICAgICAgICAKICAgICAgICByZXR1cm4gZW1iZWRkaW5nCiAgICAKICAgIGRlZiBlbWJlZF9iYXRjaChzZWxmLCB0ZXh0cyk6CiAgICAgICAgIiIiRW1iZWQgYSBiYXRjaCBvZiB0ZXh0cy4iIiIKICAgICAgICAjIEVtYmVkIG9uIENQVSB0aGVuIG1vdmUgdG8gdGFyZ2V0IGRldmljZSAoVFBVL0dQVSkKICAgICAgICB3aXRoIHRvcmNoLm5vX2dyYWQoKToKICAgICAgICAgICAgZW1iZWRkaW5ncyA9IHNlbGYubW9kZWwuZW5jb2RlKHRleHRzLCBjb252ZXJ0X3RvX3RlbnNvcj1UcnVlKQogICAgICAgICAgICBlbWJlZGRpbmdzID0gZW1iZWRkaW5ncy5jbG9uZSgpLnRvKHNlbGYuZGV2aWNlKQogICAgICAgIAogICAgICAgIGlmIHNlbGYucHJvamVjdGlvbiBpcyBub3QgTm9uZToKICAgICAgICAgICAgZW1iZWRkaW5ncyA9IHNlbGYucHJvamVjdGlvbihlbWJlZGRpbmdzLmZsb2F0KCkpCiAgICAgICAgCiAgICAgICAgcmV0dXJuIGVtYmVkZGluZ3MKCg==',
    'projects/hybrid_titans/common/titan_utils.py': 'IiIiClRpdGFuIG1vZGVsIHV0aWxpdGllcy4KU2hhcmVkIGluaXRpYWxpemF0aW9uIGFuZCBjb25maWd1cmF0aW9uIGZvciBUaXRhbk1BRy9UaXRhblJBRy4KIiIiCgppbXBvcnQgdG9yY2gKaW1wb3J0IHN5cwppbXBvcnQgb3MKCiMgQWRkIHBhcmVudCBwYXRocyBmb3IgaW1wb3J0cwpzeXMucGF0aC5pbnNlcnQoMCwgb3MucGF0aC5kaXJuYW1lKG9zLnBhdGguZGlybmFtZShvcy5wYXRoLmRpcm5hbWUob3MucGF0aC5kaXJuYW1lKG9zLnBhdGguYWJzcGF0aChfX2ZpbGVfXykpKSkpKQpzeXMucGF0aC5pbnNlcnQoMCwgb3MucGF0aC5qb2luKG9zLnBhdGguZGlybmFtZShvcy5wYXRoLmRpcm5hbWUob3MucGF0aC5kaXJuYW1lKG9zLnBhdGguZGlybmFtZShvcy5wYXRoLmFic3BhdGgoX19maWxlX18pKSkpKSwgJ3NyYycpKQoKZnJvbSBtYWluIGltcG9ydCBUaXRhbk1BRywgVGl0YW5NQUMsIFRpdGFuUkFHCgojIERldGVjdCBkZXZpY2UKREVWSUNFID0gdG9yY2guZGV2aWNlKCJjcHUiKQppZiB0b3JjaC5jdWRhLmlzX2F2YWlsYWJsZSgpOgogICAgREVWSUNFID0gdG9yY2guZGV2aWNlKCJjdWRhIikKZWxzZToKICAgIHRyeToKICAgICAgICBpbXBvcnQgdG9yY2hfeGxhLmNvcmUueGxhX21vZGVsIGFzIHhtCiAgICAgICAgREVWSUNFID0geG0ueGxhX2RldmljZSgpCiAgICAgICAgcHJpbnQoZiLinIUgVFBVIGRldGVjdGVkOiB1c2luZyBYTEEgZGV2aWNlIHtERVZJQ0V9IikKICAgIGV4Y2VwdCBJbXBvcnRFcnJvcjoKICAgICAgICBwYXNzCgoKZGVmIGNyZWF0ZV90aXRhbl9yYWcoZGltPTM4NCwgYXJjaD0iTUFHIiwgd2luZG93X3NpemU9MzIsIG1lbW9yeV9kZXB0aD0zLCAKICAgICAgICAgICAgICAgICAgICAgbnVtX3BlcnNpc3RlbnRfdG9rZW5zPTQsIHRocmVzaG9sZD0wLjAsCiAgICAgICAgICAgICAgICAgICAgIGxlYXJuaW5nX3JhdGU9MC4wMSwgZm9yZ2V0dGluZ19yYXRlPTAuMDAwMSwKICAgICAgICAgICAgICAgICAgICAgZGV2aWNlPU5vbmUpOgogICAgIiIiCiAgICBDcmVhdGUgYW5kIGNvbmZpZ3VyZSBhIFRpdGFuUkFHIGluc3RhbmNlLgogICAgCiAgICBBcmdzOgogICAgICAgIGRpbTogRW1iZWRkaW5nIGRpbWVuc2lvbgogICAgICAgIGFyY2g6IEFyY2hpdGVjdHVyZSB0eXBlICgiTUFHIiBvciAiTUFDIikKICAgICAgICB3aW5kb3dfc2l6ZTogU2xpZGluZyB3aW5kb3cgc2l6ZSBmb3IgYXR0ZW50aW9uCiAgICAgICAgbWVtb3J5X2RlcHRoOiBOdW1iZXIgb2YgbGF5ZXJzIGluIG1lbW9yeSBNTFAKICAgICAgICBudW1fcGVyc2lzdGVudF90b2tlbnM6IE51bWJlciBvZiBwZXJzaXN0ZW50IG1lbW9yeSB0b2tlbnMKICAgICAgICB0aHJlc2hvbGQ6IFN1cnByaXNlIHRocmVzaG9sZCBmb3Igc3BhcnNlIHVwZGF0ZXMKICAgICAgICBsZWFybmluZ19yYXRlOiBMZWFybmluZyByYXRlICh0aGV0YSkgZm9yIG1lbW9yeSB1cGRhdGVzCiAgICAgICAgZm9yZ2V0dGluZ19yYXRlOiBGb3JnZXR0aW5nIHJhdGUgKGFscGhhKSBmb3IgbWVtb3J5IGRlY2F5CiAgICAgICAgZGV2aWNlOiBEZXZpY2UgdG8gdXNlIChkZWZhdWx0OiBhdXRvLWRldGVjdCkKICAgIAogICAgUmV0dXJuczoKICAgICAgICBUaXRhblJBRzogQ29uZmlndXJlZCBUaXRhblJBRyBpbnN0YW5jZQogICAgIiIiCiAgICBpZiBkZXZpY2UgaXMgTm9uZToKICAgICAgICBkZXZpY2UgPSBERVZJQ0UKICAgIAogICAgaWYgYXJjaC51cHBlcigpID09ICJNQUMiOgogICAgICAgIGJhc2VfbW9kZWwgPSBUaXRhbk1BQygKICAgICAgICAgICAgZGltPWRpbSwKICAgICAgICAgICAgY2h1bmtfc2l6ZT13aW5kb3dfc2l6ZSwKICAgICAgICAgICAgaGlkZGVuX2RpbT1kaW0gKiAyLAogICAgICAgICAgICBtZW1vcnlfZGVwdGg9bWVtb3J5X2RlcHRoLAogICAgICAgICAgICBudW1fcGVyc2lzdGVudF90b2tlbnM9bnVtX3BlcnNpc3RlbnRfdG9rZW5zLAogICAgICAgICAgICB0aHJlc2hvbGQ9dGhyZXNob2xkCiAgICAgICAgKQogICAgZWxzZToKICAgICAgICBiYXNlX21vZGVsID0gVGl0YW5NQUcoCiAgICAgICAgICAgIGRpbT1kaW0sCiAgICAgICAgICAgIHdpbmRvd19zaXplPXdpbmRvd19zaXplLAogICAgICAgICAgICBoaWRkZW5fZGltPWRpbSAqIDIsCiAgICAgICAgICAgIG1lbW9yeV9kZXB0aD1tZW1vcnlfZGVwdGgsCiAgICAgICAgICAgIG51bV9wZXJzaXN0ZW50X3Rva2Vucz1udW1fcGVyc2lzdGVudF90b2tlbnMsCiAgICAgICAgICAgIHRocmVzaG9sZD10aHJlc2hvbGQKICAgICAgICApCiAgICAKICAgICMgTW92ZSBtb2RlbCB0byBkZXZpY2UKICAgIGJhc2VfbW9kZWwgPSBiYXNlX21vZGVsLnRvKGRldmljZSkKICAgIAogICAgIyBDb25maWd1cmUgbWVtb3J5IHBhcmFtZXRlcnMKICAgIHdpdGggdG9yY2gubm9fZ3JhZCgpOgogICAgICAgIGJhc2VfbW9kZWwubHRtLnRoZXRhLmZpbGxfKGxlYXJuaW5nX3JhdGUpCiAgICAgICAgYmFzZV9tb2RlbC5sdG0uYWxwaGEuZmlsbF8oZm9yZ2V0dGluZ19yYXRlKQogICAgCiAgICB0aXRhbl9yYWcgPSBUaXRhblJBRyhiYXNlX21vZGVsKQogICAgCiAgICBpZiBzdHIoZGV2aWNlKS5zdGFydHN3aXRoKCd4bGEnKToKICAgICAgICBkZXZpY2VfbmFtZSA9ICJUUFUiCiAgICBlbGlmIGRldmljZS50eXBlID09ICJjdWRhIjoKICAgICAgICBkZXZpY2VfbmFtZSA9ICJHUFUiCiAgICBlbHNlOgogICAgICAgIGRldmljZV9uYW1lID0gIkNQVSIKICAgIHByaW50KGYi4pyFIENyZWF0ZWQgVGl0YW5SQUcgd2l0aCB7YXJjaH0gYXJjaGl0ZWN0dXJlIChkaW09e2RpbX0pIG9uIHtkZXZpY2VfbmFtZX0iKQogICAgCiAgICByZXR1cm4gdGl0YW5fcmFnCgoKZGVmIGRpZ2VzdF9kb2N1bWVudCh0aXRhbl9yYWcsIGVtYmVkZGVyLCB0ZXh0LCBlcG9jaHM9NTApOgogICAgIiIiCiAgICBEaWdlc3QgYSBkb2N1bWVudCBpbnRvIFRpdGFuUkFHIG1lbW9yeS4KICAgIAogICAgQXJnczoKICAgICAgICB0aXRhbl9yYWc6IFRpdGFuUkFHIGluc3RhbmNlCiAgICAgICAgZW1iZWRkZXI6IEVtYmVkZGVyIHRvIGNvbnZlcnQgdGV4dCB0byBlbWJlZGRpbmdzCiAgICAgICAgdGV4dDogRG9jdW1lbnQgdGV4dCAod2lsbCBiZSBzcGxpdCBieSBwYXJhZ3JhcGhzKQogICAgICAgIGVwb2NoczogTnVtYmVyIG9mIGRpZ2VzdGlvbiBlcG9jaHMKICAgIAogICAgUmV0dXJuczoKICAgICAgICB0dXBsZTogKHBhcmFncmFwaHMsIHBhcmFncmFwaF9lbWJlZGRpbmdzKQogICAgIiIiCiAgICAjIFBhcnNlIGludG8gcGFyYWdyYXBocwogICAgcmF3X3BhcmFncmFwaHMgPSB0ZXh0LnNwbGl0KCdcblxuJykKICAgIHBhcmFncmFwaHMgPSBbXQogICAgZm9yIHBhcmEgaW4gcmF3X3BhcmFncmFwaHM6CiAgICAgICAgY2xlYW5lZCA9ICcgJy5qb2luKGxpbmUuc3RyaXAoKSBmb3IgbGluZSBpbiBwYXJhLnNwbGl0KCdcbicpIGlmIGxpbmUuc3RyaXAoKSkKICAgICAgICBpZiBjbGVhbmVkIGFuZCBsZW4oY2xlYW5lZCkgPiAxMDoKICAgICAgICAgICAgcGFyYWdyYXBocy5hcHBlbmQoY2xlYW5lZCkKICAgIAogICAgIyBFbWJlZCBwYXJhZ3JhcGhzCiAgICBhbGxfZW1iZWRkaW5ncyA9IHRvcmNoLmNhdChbZW1iZWRkZXIocCkgZm9yIHAgaW4gcGFyYWdyYXBoc10sIGRpbT0xKQogICAgCiAgICAjIERpZ2VzdCBpbnRvIG1lbW9yeQogICAgcHJpbnQoZiLwn6egIERpZ2VzdGluZyB7bGVuKHBhcmFncmFwaHMpfSBwYXJhZ3JhcGhzICh7ZXBvY2hzfSBlcG9jaHMpLi4uIikKICAgIGZvciBlcG9jaCBpbiByYW5nZShlcG9jaHMpOgogICAgICAgIHRpdGFuX3JhZy5kaWdlc3Rfa25vd2xlZGdlKGFsbF9lbWJlZGRpbmdzKQogICAgICAgIGlmIChlcG9jaCArIDEpICUgMTAgPT0gMDoKICAgICAgICAgICAgcHJpbnQoZiIgICBFcG9jaCB7ZXBvY2ggKyAxfS97ZXBvY2hzfSBjb21wbGV0ZWQiKQogICAgCiAgICAjIENvbXB1dGUgcGFyYWdyYXBoIGVtYmVkZGluZ3MgZm9yIHJldHJpZXZhbAogICAgcGFyYWdyYXBoX2VtYmVkZGluZ3MgPSBlbWJlZGRlci5lbWJlZF9iYXRjaChwYXJhZ3JhcGhzKQogICAgCiAgICByZXR1cm4gcGFyYWdyYXBocywgcGFyYWdyYXBoX2VtYmVkZGluZ3MK',
    'projects/hybrid_titans/common/llm_utils.py': 'IiIiCkxMTSB1dGlsaXRpZXMgZm9yIHRleHQgZ2VuZXJhdGlvbi4KU2hhcmVkIEZsYW4tVDUgbG9hZGluZyBhbmQgZ2VuZXJhdGlvbiBmdW5jdGlvbnMuCiIiIgoKaW1wb3J0IHRvcmNoCgoKY2xhc3MgRmxhblQ1R2VuZXJhdG9yOgogICAgIiIiRmxhbi1UNSBiYXNlZCB0ZXh0IGdlbmVyYXRvci4iIiIKICAgIAogICAgZGVmIF9faW5pdF9fKHNlbGYsIG1vZGVsX25hbWU9Imdvb2dsZS9mbGFuLXQ1LXhsIiwgZGV2aWNlPU5vbmUpOgogICAgICAgIGZyb20gdHJhbnNmb3JtZXJzIGltcG9ydCBUNUZvckNvbmRpdGlvbmFsR2VuZXJhdGlvbiwgVDVUb2tlbml6ZXIKICAgICAgICAKICAgICAgICBzZWxmLmRldmljZSA9IGRldmljZQogICAgICAgIHNlbGYudG9rZW5pemVyID0gVDVUb2tlbml6ZXIuZnJvbV9wcmV0cmFpbmVkKG1vZGVsX25hbWUpCiAgICAgICAgc2VsZi5sbG0gPSBUNUZvckNvbmRpdGlvbmFsR2VuZXJhdGlvbi5mcm9tX3ByZXRyYWluZWQobW9kZWxfbmFtZSkKICAgICAgICAKICAgICAgICBpZiBzZWxmLmRldmljZToKICAgICAgICAgICAgc2VsZi5sbG0gPSBzZWxmLmxsbS50byhzZWxmLmRldmljZSkKICAgICAgICAgICAgCiAgICAgICAgc2VsZi5tb2RlbF9uYW1lID0gbW9kZWxfbmFtZQogICAgICAgIAogICAgICAgIGRldmljZV9pbmZvID0gZiIgb24ge3NlbGYuZGV2aWNlfSIgaWYgc2VsZi5kZXZpY2UgZWxzZSAiIgogICAgICAgIHByaW50KGYi4pyFIExvYWRlZCBMTE06IHttb2RlbF9uYW1lfXtkZXZpY2VfaW5mb30iKQogICAgCiAgICBkZWYgZ2VuZXJhdGUoc2VsZiwgcHJvbXB0LCBtYXhfbmV3X3Rva2Vucz0xMDAsIG51bV9iZWFtcz00KToKICAgICAgICAiIiIKICAgICAgICBHZW5lcmF0ZSB0ZXh0IGZyb20gYSBwcm9tcHQuCiAgICAgICAgCiAgICAgICAgQXJnczoKICAgICAgICAgICAgcHJvbXB0OiBJbnB1dCBwcm9tcHQKICAgICAgICAgICAgbWF4X25ld190b2tlbnM6IE1heGltdW0gdG9rZW5zIHRvIGdlbmVyYXRlCiAgICAgICAgICAgIG51bV9iZWFtczogQmVhbSBzZWFyY2ggd2lkdGgKICAgICAgICAKICAgICAgICBSZXR1cm5zOgogICAgICAgICAgICBzdHI6IEdlbmVyYXRlZCB0ZXh0CiAgICAgICAgIiIiCiAgICAgICAgaW5wdXRzID0gc2VsZi50b2tlbml6ZXIoCiAgICAgICAgICAgIHByb21wdCwKICAgICAgICAgICAgcmV0dXJuX3RlbnNvcnM9InB0IiwKICAgICAgICAgICAgdHJ1bmNhdGlvbj1UcnVlLAogICAgICAgICAgICBtYXhfbGVuZ3RoPTEwMjQKICAgICAgICApCiAgICAgICAgCiAgICAgICAgIyBNb3ZlIGlucHV0cyB0byB0aGUgc2FtZSBkZXZpY2UgYXMgdGhlIG1vZGVsCiAgICAgICAgaWYgc2VsZi5kZXZpY2U6CiAgICAgICAgICAgIGlucHV0cyA9IHtrOiB2LnRvKHNlbGYuZGV2aWNlKSBmb3IgaywgdiBpbiBpbnB1dHMuaXRlbXMoKX0KICAgICAgICAKICAgICAgICBpZiBzdHIoc2VsZi5kZXZpY2UpLnN0YXJ0c3dpdGgoJ3hsYScpOgogICAgICAgICAgICAjIEZvcmNlIGdyZWVkeSBzZWFyY2ggb24gVFBVIHRvIGF2b2lkIGJlYW1fc2VhcmNoIGhhbmdzICh0b3JjaC5pc2luIGlzc3VlcykKICAgICAgICAgICAgbnVtX2JlYW1zID0gMQogICAgICAgICAgICAKICAgICAgICB3aXRoIHRvcmNoLm5vX2dyYWQoKToKICAgICAgICAgICAgb3V0cHV0cyA9IHNlbGYubGxtLmdlbmVyYXRlKAogICAgICAgICAgICAgICAgKippbnB1dHMsCiAgICAgICAgICAgICAgICBtYXhfbmV3X3Rva2Vucz1tYXhfbmV3X3Rva2VucywKICAgICAgICAgICAgICAgIG51bV9iZWFtcz1udW1fYmVhbXMsCiAgICAgICAgICAgICAgICBlYXJseV9zdG9wcGluZz0obnVtX2JlYW1zID4gMSksCiAgICAgICAgICAgICAgICBkb19zYW1wbGU9RmFsc2UsCiAgICAgICAgICAgICAgICBub19yZXBlYXRfbmdyYW1fc2l6ZT0yIGlmIG51bV9iZWFtcyA+IDEgZWxzZSAwCiAgICAgICAgICAgICkKICAgICAgICAKICAgICAgICBhbnN3ZXIgPSBzZWxmLnRva2VuaXplci5kZWNvZGUob3V0cHV0c1swXSwgc2tpcF9zcGVjaWFsX3Rva2Vucz1UcnVlKQogICAgICAgIHJldHVybiBhbnN3ZXIKICAgIAogICAgZGVmIGdlbmVyYXRlX3FhKHNlbGYsIHF1ZXN0aW9uLCBjb250ZXh0LCBtYXhfbmV3X3Rva2Vucz0xMDAsIG11bHRpaG9wPUZhbHNlKToKICAgICAgICAiIiIKICAgICAgICBHZW5lcmF0ZSBhbnN3ZXIgZm9yIGEgcXVlc3Rpb24gZ2l2ZW4gY29udGV4dC4KICAgICAgICAKICAgICAgICBBcmdzOgogICAgICAgICAgICBxdWVzdGlvbjogVGhlIHF1ZXN0aW9uCiAgICAgICAgICAgIGNvbnRleHQ6IENvbnRleHQvcGFzc2FnZSB0byBhbnN3ZXIgZnJvbQogICAgICAgICAgICBtYXhfbmV3X3Rva2VuczogTWF4aW11bSB0b2tlbnMgdG8gZ2VuZXJhdGUKICAgICAgICAgICAgbXVsdGlob3A6IElmIFRydWUsIHVzZSBtdWx0aS1ob3AgcmVhc29uaW5nIHByb21wdCAoZm9yIEhvdHBvdFFBKS4KICAgICAgICAgICAgICAgICAgICAgIElmIEZhbHNlLCB1c2UgZXh0cmFjdGl2ZSBwcm9tcHQgKGZvciBTUXVBRCkuCiAgICAgICAgCiAgICAgICAgUmV0dXJuczoKICAgICAgICAgICAgdHVwbGU6IChhbnN3ZXIsIHByb21wdCkKICAgICAgICAiIiIKICAgICAgICBpZiBtdWx0aWhvcDoKICAgICAgICAgICAgcHJvbXB0ID0gZiIiIkFuc3dlciB0aGUgcXVlc3Rpb24gYmFzZWQgb24gdGhlIGNvbnRleHQgcHJvdmlkZWQuCkV4dHJhY3Qgc3BlY2lmaWMgZGV0YWlsczogZXhhY3QgbnVtYmVycywgbmFtZXMsIGRhdGVzLCBhbmQgdGVjaG5pY2FsIHRlcm1zLgpQcm92aWRlIGEgY29uY2lzZSwgZGlyZWN0IGFuc3dlci4KCkNvbnRleHQ6Cntjb250ZXh0fQoKUXVlc3Rpb246IHtxdWVzdGlvbn0KCkFuc3dlcjoiIiIKICAgICAgICBlbHNlOgogICAgICAgICAgICBwcm9tcHQgPSBmIiIiQW5zd2VyIHRoZSBxdWVzdGlvbiBiYXNlZCBvbiB0aGUgY29udGV4dCBwcm92aWRlZC4KRXh0cmFjdCBzcGVjaWZpYyBkZXRhaWxzOiBleGFjdCBudW1iZXJzLCBuYW1lcywgZGF0ZXMsIGFuZCB0ZWNobmljYWwgdGVybXMuClByb3ZpZGUgdGhlIG1vc3QgcmVsZXZhbnQgYW5zd2VyIGZyb20gdGhlIGNvbnRleHQsIGV2ZW4gaWYgcGFydGlhbC4KCkNvbnRleHQ6Cntjb250ZXh0fQoKUXVlc3Rpb246IHtxdWVzdGlvbn0KCkFuc3dlcjoiIiIKICAgICAgICAKICAgICAgICBhbnN3ZXIgPSBzZWxmLmdlbmVyYXRlKHByb21wdCwgbWF4X25ld190b2tlbnMpCiAgICAgICAgcmV0dXJuIGFuc3dlciwgcHJvbXB0Cg==',
    'projects/hybrid_titans/common/text_utils.py': 'IiIiClRleHQgdXRpbGl0aWVzIGZvciBkb2N1bWVudCBwcm9jZXNzaW5nLgpTaGFyZWQgY2h1bmtpbmcgYW5kIHRleHQgcHJvY2Vzc2luZyBmdW5jdGlvbnMuCiIiIgoKaW1wb3J0IHJlCgoKZGVmIHNwbGl0X2ludG9fY2h1bmtzKHRleHQsIG1pbl9sZW5ndGg9MzAsIHNlbnRlbmNlc19wZXJfY2h1bms9Mywgb3ZlcmxhcF9zZW50ZW5jZXM9MSk6CiAgICAiIiIKICAgIFNwbGl0IHRleHQgaW50byBzZW1hbnRpYyBjaHVua3MgZm9yIHJldHJpZXZhbCB3aXRoIG92ZXJsYXBwaW5nIHdpbmRvd3MuCiAgICAKICAgIFN0cmF0ZWd5OgogICAgLSBIZWFkZXJzIGFyZSBncm91cGVkIHdpdGggdGhlaXIgZm9sbG93aW5nIHNlbnRlbmNlcwogICAgLSBDaHVua3Mgb3ZlcmxhcCBieSAxIHNlbnRlbmNlIHRvIGF2b2lkIGJvdW5kYXJ5IGluZm9ybWF0aW9uIGxvc3MKICAgIC0gRWFjaCBjaHVuayBjb250YWlucyAzIHNlbnRlbmNlcyBmb3IgYmV0dGVyIGNvbnRleHQKICAgIAogICAgQXJnczoKICAgICAgICB0ZXh0OiBJbnB1dCB0ZXh0IChjYW4gY29udGFpbiBtYXJrZG93biBoZWFkZXJzKQogICAgICAgIG1pbl9sZW5ndGg6IE1pbmltdW0gY2h1bmsgbGVuZ3RoIHRvIGtlZXAKICAgICAgICBzZW50ZW5jZXNfcGVyX2NodW5rOiBOdW1iZXIgb2Ygc2VudGVuY2VzIHBlciBjaHVuayAoZGVmYXVsdDogMykKICAgICAgICBvdmVybGFwX3NlbnRlbmNlczogTnVtYmVyIG9mIHNlbnRlbmNlcyB0byBvdmVybGFwIGJldHdlZW4gY2h1bmtzIChkZWZhdWx0OiAxKQogICAgICAgIAogICAgUmV0dXJuczoKICAgICAgICBMaXN0IG9mIHRleHQgY2h1bmtzCiAgICAiIiIKICAgIGxpbmVzID0gdGV4dC5zcGxpdCgnXG4nKQogICAgY2h1bmtzID0gW10KICAgIGN1cnJlbnRfaGVhZGVyID0gIiIKICAgIGFsbF9zZW50ZW5jZXMgPSBbXSAgIyBDb2xsZWN0IGFsbCBzZW50ZW5jZXMgZmlyc3QKICAgIAogICAgIyBGaXJzdCBwYXNzOiBjb2xsZWN0IGFsbCBzZW50ZW5jZXMgd2l0aCB0aGVpciBoZWFkZXJzCiAgICBmb3IgbGluZSBpbiBsaW5lczoKICAgICAgICBsaW5lID0gbGluZS5zdHJpcCgpCiAgICAgICAgaWYgbm90IGxpbmU6CiAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgCiAgICAgICAgIyBDaGVjayBpZiBpdCdzIGEgaGVhZGVyCiAgICAgICAgaWYgbGluZS5zdGFydHN3aXRoKCcjJyk6CiAgICAgICAgICAgICMgSWYgd2UgaGF2ZSBhY2N1bXVsYXRlZCBzZW50ZW5jZXMsIGNyZWF0ZSBjaHVuayBiZWZvcmUgbmV3IGhlYWRlcgogICAgICAgICAgICBpZiBhbGxfc2VudGVuY2VzOgogICAgICAgICAgICAgICAgX2NyZWF0ZV9jaHVua3NfZnJvbV9zZW50ZW5jZXMoCiAgICAgICAgICAgICAgICAgICAgY2h1bmtzLCBjdXJyZW50X2hlYWRlciwgYWxsX3NlbnRlbmNlcywgCiAgICAgICAgICAgICAgICAgICAgc2VudGVuY2VzX3Blcl9jaHVuaywgb3ZlcmxhcF9zZW50ZW5jZXMsIG1pbl9sZW5ndGgKICAgICAgICAgICAgICAgICkKICAgICAgICAgICAgICAgIGFsbF9zZW50ZW5jZXMgPSBbXQogICAgICAgICAgICBjdXJyZW50X2hlYWRlciA9IGxpbmUKICAgICAgICAgICAgY29udGludWUKICAgICAgICAKICAgICAgICAjIEhhbmRsZSByZWd1bGFyIGNvbnRlbnQgLSBzcGxpdCBpbnRvIHNlbnRlbmNlcwogICAgICAgICMgUHJvdGVjdCBhYmJyZXZpYXRpb25zCiAgICAgICAgcHJvdGVjdGVkID0gbGluZS5yZXBsYWNlKCdEci4nLCAnRHLCpycpLnJlcGxhY2UoJ01yLicsICdNcsKnJykucmVwbGFjZSgnTXMuJywgJ01zwqcnKQogICAgICAgIHByb3RlY3RlZCA9IHByb3RlY3RlZC5yZXBsYWNlKCdldGMuJywgJ2V0Y8KnJykucmVwbGFjZSgnZS5nLicsICdlZ8KnJykucmVwbGFjZSgnaS5lLicsICdpZcKnJykKICAgICAgICAKICAgICAgICAjIFNwbGl0IG9uIHNlbnRlbmNlIGJvdW5kYXJpZXMKICAgICAgICBwYXJ0cyA9IHJlLnNwbGl0KHInKD88PVsuIT9dKVxzKycsIHByb3RlY3RlZCkKICAgICAgICAKICAgICAgICBmb3IgcGFydCBpbiBwYXJ0czoKICAgICAgICAgICAgcGFydCA9IHBhcnQuc3RyaXAoKQogICAgICAgICAgICAjIFJlc3RvcmUgYWJicmV2aWF0aW9ucwogICAgICAgICAgICBwYXJ0ID0gcGFydC5yZXBsYWNlKCdEcsKnJywgJ0RyLicpLnJlcGxhY2UoJ01ywqcnLCAnTXIuJykucmVwbGFjZSgnTXPCpycsICdNcy4nKQogICAgICAgICAgICBwYXJ0ID0gcGFydC5yZXBsYWNlKCdldGPCpycsICdldGMuJykucmVwbGFjZSgnZWfCpycsICdlLmcuJykucmVwbGFjZSgnaWXCpycsICdpLmUuJykKICAgICAgICAgICAgCiAgICAgICAgICAgIGlmIGxlbihwYXJ0KSA+PSAxMDogICMgTWluaW11bSBzZW50ZW5jZSBsZW5ndGgKICAgICAgICAgICAgICAgIGFsbF9zZW50ZW5jZXMuYXBwZW5kKHBhcnQpCiAgICAKICAgICMgRmx1c2ggcmVtYWluaW5nIHNlbnRlbmNlcwogICAgaWYgYWxsX3NlbnRlbmNlcyBvciBjdXJyZW50X2hlYWRlcjoKICAgICAgICBfY3JlYXRlX2NodW5rc19mcm9tX3NlbnRlbmNlcygKICAgICAgICAgICAgY2h1bmtzLCBjdXJyZW50X2hlYWRlciwgYWxsX3NlbnRlbmNlcywgCiAgICAgICAgICAgIHNlbnRlbmNlc19wZXJfY2h1bmssIG92ZXJsYXBfc2VudGVuY2VzLCBtaW5fbGVuZ3RoCiAgICAgICAgKQogICAgCiAgICByZXR1cm4gY2h1bmtzCgoKZGVmIF9jcmVhdGVfY2h1bmtzX2Zyb21fc2VudGVuY2VzKGNodW5rcywgaGVhZGVyLCBzZW50ZW5jZXMsIAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIHNlbnRlbmNlc19wZXJfY2h1bmssIG92ZXJsYXBfc2VudGVuY2VzLCBtaW5fbGVuZ3RoKToKICAgICIiIkhlbHBlcjogQ3JlYXRlIG92ZXJsYXBwaW5nIGNodW5rcyBmcm9tIGEgbGlzdCBvZiBzZW50ZW5jZXMuIiIiCiAgICBpZiBub3Qgc2VudGVuY2VzOgogICAgICAgIGlmIGhlYWRlciBhbmQgbGVuKGhlYWRlcikgPiA1OgogICAgICAgICAgICBjaHVua3MuYXBwZW5kKGhlYWRlcikKICAgICAgICByZXR1cm4KICAgIAogICAgIyBJZiB3ZSBoYXZlIGZld2VyIHNlbnRlbmNlcyB0aGFuIGNodW5rIHNpemUsIGNyZWF0ZSBvbmUgY2h1bmsKICAgIGlmIGxlbihzZW50ZW5jZXMpIDw9IHNlbnRlbmNlc19wZXJfY2h1bms6CiAgICAgICAgY29udGVudCA9ICIgIi5qb2luKHNlbnRlbmNlcykKICAgICAgICBpZiBoZWFkZXI6CiAgICAgICAgICAgIGNodW5rID0gZiJ7aGVhZGVyfSB7Y29udGVudH0iCiAgICAgICAgZWxzZToKICAgICAgICAgICAgY2h1bmsgPSBjb250ZW50CiAgICAgICAgaWYgbGVuKGNodW5rKSA+PSBtaW5fbGVuZ3RoOgogICAgICAgICAgICBjaHVua3MuYXBwZW5kKGNodW5rKQogICAgICAgIHJldHVybgogICAgCiAgICAjIENyZWF0ZSBvdmVybGFwcGluZyBjaHVua3MgdXNpbmcgc2xpZGluZyB3aW5kb3cKICAgIHN0ZXAgPSBzZW50ZW5jZXNfcGVyX2NodW5rIC0gb3ZlcmxhcF9zZW50ZW5jZXMKICAgIHN0ZXAgPSBtYXgoMSwgc3RlcCkgICMgRW5zdXJlIGF0IGxlYXN0IDEgc3RlcAogICAgCiAgICBmb3IgaSBpbiByYW5nZSgwLCBsZW4oc2VudGVuY2VzKSwgc3RlcCk6CiAgICAgICAgd2luZG93ID0gc2VudGVuY2VzW2k6aSArIHNlbnRlbmNlc19wZXJfY2h1bmtdCiAgICAgICAgaWYgbm90IHdpbmRvdzoKICAgICAgICAgICAgYnJlYWsKICAgICAgICAgICAgCiAgICAgICAgY29udGVudCA9ICIgIi5qb2luKHdpbmRvdykKICAgICAgICAKICAgICAgICAjIEluY2x1ZGUgaGVhZGVyIG9ubHkgaW4gZmlyc3QgY2h1bmsgb2Ygc2VjdGlvbgogICAgICAgIGlmIGkgPT0gMCBhbmQgaGVhZGVyOgogICAgICAgICAgICBjaHVuayA9IGYie2hlYWRlcn0ge2NvbnRlbnR9IgogICAgICAgIGVsc2U6CiAgICAgICAgICAgIGNodW5rID0gY29udGVudAogICAgICAgICAgICAKICAgICAgICBpZiBsZW4oY2h1bmspID49IG1pbl9sZW5ndGg6CiAgICAgICAgICAgIGNodW5rcy5hcHBlbmQoY2h1bmspCiAgICAgICAgCiAgICAgICAgIyBTdG9wIGlmIHdlJ3ZlIHByb2Nlc3NlZCBhbGwgc2VudGVuY2VzCiAgICAgICAgaWYgaSArIHNlbnRlbmNlc19wZXJfY2h1bmsgPj0gbGVuKHNlbnRlbmNlcyk6CiAgICAgICAgICAgIGJyZWFrCgoKZGVmIHNpbXBsZV9zcGxpdCh0ZXh0KToKICAgICIiIlNpbXBsZSBsaW5lLWJ5LWxpbmUgc3BsaXQgKGZvciBjb21wYXJpc29uKS4iIiIKICAgIHJldHVybiBbbGluZS5zdHJpcCgpIGZvciBsaW5lIGluIHRleHQuc3BsaXQoJ1xuJykgaWYgbGluZS5zdHJpcCgpIGFuZCBsZW4obGluZS5zdHJpcCgpKSA+IDEwXQo=',
    'projects/hybrid_titans/common/eval_utils.py': 'IiIiCkV2YWx1YXRpb24gdXRpbGl0aWVzIGZvciBhbnN3ZXIgcXVhbGl0eSBhc3Nlc3NtZW50LgpTdGFuZGFyZCBTUXVBRC1zdHlsZSBFeGFjdCBNYXRjaCAoRU0pIGFuZCB0b2tlbi1sZXZlbCBGMSBTY29yZS4KIiIiCgppbXBvcnQgcmUKaW1wb3J0IHN0cmluZwpmcm9tIGNvbGxlY3Rpb25zIGltcG9ydCBDb3VudGVyCgoKZGVmIG5vcm1hbGl6ZV9hbnN3ZXIocyk6CiAgICAiIiJOb3JtYWxpemUgYW5zd2VyIHN0cmluZyBmb3IgRU0vRjEgY29tcGFyaXNvbi4KICAgIAogICAgRm9sbG93cyB0aGUgU1F1QUQgZXZhbHVhdGlvbiBzY3JpcHQgY29udmVudGlvbjoKICAgIGxvd2VyY2FzZSDihpIgcmVtb3ZlIGFydGljbGVzIOKGkiByZW1vdmUgcHVuY3R1YXRpb24g4oaSIGNvbGxhcHNlIHdoaXRlc3BhY2UuCiAgICAiIiIKICAgIHMgPSBzLmxvd2VyKCkKICAgICMgUmVtb3ZlIGFydGljbGVzCiAgICBzID0gcmUuc3ViKHInXGIoYXxhbnx0aGUpXGInLCAnICcsIHMpCiAgICAjIFJlbW92ZSBwdW5jdHVhdGlvbgogICAgcyA9IHMudHJhbnNsYXRlKHN0ci5tYWtldHJhbnMoJycsICcnLCBzdHJpbmcucHVuY3R1YXRpb24pKQogICAgIyBDb2xsYXBzZSB3aGl0ZXNwYWNlCiAgICBzID0gJyAnLmpvaW4ocy5zcGxpdCgpKQogICAgcmV0dXJuIHMKCgpkZWYgY29tcHV0ZV9lbShleHBlY3RlZCwgcHJlZGljdGVkKToKICAgICIiIkV4YWN0IE1hdGNoOiAxIGlmIG5vcm1hbGl6ZWQgc3RyaW5ncyBhcmUgaWRlbnRpY2FsLCBlbHNlIDAuIiIiCiAgICByZXR1cm4gaW50KG5vcm1hbGl6ZV9hbnN3ZXIoZXhwZWN0ZWQpID09IG5vcm1hbGl6ZV9hbnN3ZXIocHJlZGljdGVkKSkKCgpkZWYgY29tcHV0ZV9mMShleHBlY3RlZCwgcHJlZGljdGVkKToKICAgICIiIlRva2VuLWxldmVsIEYxIFNjb3JlLgogICAgCiAgICBDb21wdXRlcyBwcmVjaXNpb24gYW5kIHJlY2FsbCBvdmVyIHNoYXJlZCB0b2tlbnMgYmV0d2VlbgogICAgbm9ybWFsaXplZCBleHBlY3RlZCBhbmQgcHJlZGljdGVkIGFuc3dlcnMsIHJldHVybnMgdGhlaXIKICAgIGhhcm1vbmljIG1lYW4uCiAgICAKICAgIFJldHVybnM6CiAgICAgICAgZmxvYXQ6IEYxIHNjb3JlIGluIFswLCAxXQogICAgIiIiCiAgICBleHBlY3RlZF90b2tlbnMgPSBub3JtYWxpemVfYW5zd2VyKGV4cGVjdGVkKS5zcGxpdCgpCiAgICBwcmVkaWN0ZWRfdG9rZW5zID0gbm9ybWFsaXplX2Fuc3dlcihwcmVkaWN0ZWQpLnNwbGl0KCkKICAgIAogICAgaWYgbm90IGV4cGVjdGVkX3Rva2VucyBhbmQgbm90IHByZWRpY3RlZF90b2tlbnM6CiAgICAgICAgcmV0dXJuIDEuMAogICAgaWYgbm90IGV4cGVjdGVkX3Rva2VucyBvciBub3QgcHJlZGljdGVkX3Rva2VuczoKICAgICAgICByZXR1cm4gMC4wCiAgICAKICAgIGNvbW1vbiA9IENvdW50ZXIoZXhwZWN0ZWRfdG9rZW5zKSAmIENvdW50ZXIocHJlZGljdGVkX3Rva2VucykKICAgIG51bV9jb21tb24gPSBzdW0oY29tbW9uLnZhbHVlcygpKQogICAgCiAgICBpZiBudW1fY29tbW9uID09IDA6CiAgICAgICAgcmV0dXJuIDAuMAogICAgCiAgICBwcmVjaXNpb24gPSBudW1fY29tbW9uIC8gbGVuKHByZWRpY3RlZF90b2tlbnMpCiAgICByZWNhbGwgPSBudW1fY29tbW9uIC8gbGVuKGV4cGVjdGVkX3Rva2VucykKICAgIGYxID0gMiAqIHByZWNpc2lvbiAqIHJlY2FsbCAvIChwcmVjaXNpb24gKyByZWNhbGwpCiAgICByZXR1cm4gZjEK',
    'projects/hybrid_titans/baselines.py': '...',
    'projects/hybrid_titans/compare_baselines.py': '...'
    ,'is_RAG_able/rag_infer.py': 'aW1wb3J0IG9zCmltcG9ydCBqc29uCmltcG9ydCB0b3JjaAppbXBvcnQgdG9yY2gubm4gYXMgbm4KaW1wb3J0IHRvcmNoLm5uLmZ1bmN0aW9uYWwgYXMgRgpmcm9tIHNlbnRlbmNlX3RyYW5zZm9ybWVycyBpbXBvcnQgU2VudGVuY2VUcmFuc2Zvcm1lcgppbXBvcnQgbnVtcHkgYXMgbnAKZnJvbSBza2xlYXJuLmZlYXR1cmVfZXh0cmFjdGlvbi50ZXh0IGltcG9ydCBUZmlkZlZlY3Rvcml6ZXIKZnJvbSBza2xlYXJuLnByZXByb2Nlc3NpbmcgaW1wb3J0IExhYmVsQmluYXJpemVyCmZyb20gc2tsZWFybi5tZXRyaWNzLnBhaXJ3aXNlIGltcG9ydCBjb3NpbmVfc2ltaWxhcml0eQoKIyBDb25zdGFudHMgYmFzZWQgb24gdGhlIHNhdmVkIG1vZGVsJ3MgbWV0YQpJTlBVVF9ESU0gPSA0MDUgICMgRGVyaXZlZCBmcm9tIG1vZGVsIHRyYWluaW5nOiAzODQoZW1iKSArIDE0KG51bSkgKyA3KHR5cGUpCkhJRERFTl9ESU0gPSAyNTYKCmNsYXNzIE1MUENsYXNzaWZpZXIobm4uTW9kdWxlKToKICAgIGRlZiBfX2luaXRfXyhzZWxmLCBpbnB1dF9kaW0sIGhpZGRlbl9kaW09MjA0OCwgZHJvcG91dD0wLjIpOgogICAgICAgIHN1cGVyKCkuX19pbml0X18oKQogICAgICAgIHNlbGYuZmMxID0gbm4uTGluZWFyKGlucHV0X2RpbSwgaGlkZGVuX2RpbSkKICAgICAgICBzZWxmLmJuMSA9IG5uLkJhdGNoTm9ybTFkKGhpZGRlbl9kaW0pCiAgICAgICAgc2VsZi5mYzIgPSBubi5MaW5lYXIoaGlkZGVuX2RpbSwgaGlkZGVuX2RpbSkKICAgICAgICBzZWxmLmJuMiA9IG5uLkJhdGNoTm9ybTFkKGhpZGRlbl9kaW0pCiAgICAgICAgc2VsZi5mYzMgPSBubi5MaW5lYXIoaGlkZGVuX2RpbSwgaGlkZGVuX2RpbSAvLyAyKQogICAgICAgIHNlbGYuYm4zID0gbm4uQmF0Y2hOb3JtMWQoaGlkZGVuX2RpbSAvLyAyKQogICAgICAgIHNlbGYuZmM0ID0gbm4uTGluZWFyKGhpZGRlbl9kaW0gLy8gMiwgaGlkZGVuX2RpbSAvLyA0KQogICAgICAgIHNlbGYuYm40ID0gbm4uQmF0Y2hOb3JtMWQoaGlkZGVuX2RpbSAvLyA0KQogICAgICAgIHNlbGYuZmM1ID0gbm4uTGluZWFyKGhpZGRlbl9kaW0gLy8gNCwgMSkKICAgICAgICBzZWxmLmRyb3BvdXQgPSBubi5Ecm9wb3V0KGRyb3BvdXQpCiAgICAgICAgCiAgICBkZWYgZm9yd2FyZChzZWxmLCB4KToKICAgICAgICB4MSA9IEYuZ2VsdShzZWxmLmJuMShzZWxmLmZjMSh4KSkpCiAgICAgICAgeDIgPSBGLmdlbHUoc2VsZi5ibjIoc2VsZi5mYzIoeDEpKSkgKyB4MSAgIyBSZXNpZHVhbAogICAgICAgIHgzID0gRi5nZWx1KHNlbGYuYm4zKHNlbGYuZmMzKHgyKSkpCiAgICAgICAgeDQgPSBGLmdlbHUoc2VsZi5ibjQoc2VsZi5mYzQoeDMpKSkKICAgICAgICB4NCA9IHNlbGYuZHJvcG91dCh4NCkKICAgICAgICBvdXQgPSBzZWxmLmZjNSh4NCkKICAgICAgICByZXR1cm4gb3V0LnNxdWVlemUoLTEpCgoKY2xhc3MgQmFzZUNvbmZpZGVuY2VTY29yZXI6CiAgICAiIiJCYXNlIGNsYXNzIHdpdGggc2hhcmVkIGZlYXR1cmUgZXh0cmFjdGlvbiBsb2dpYyBmb3IgUkFHIGNvbmZpZGVuY2Ugc2NvcmluZy4KICAgIAogICAgU3ViY2xhc3NlcyBvbmx5IG5lZWQgdG8gaW1wbGVtZW50IG1vZGVsIGxvYWRpbmcgYW5kIHRoZSBmaW5hbCBwcmVkaWN0aW9uIHN0ZXAuCiAgICAiIiIKICAgIAogICAgZGVmIF9faW5pdF9fKHNlbGYsIGVtYmVkZGVyX25hbWU9ImFsbC1NaW5pTE0tTDYtdjIiLCBkZXZpY2U9Tm9uZSk6CiAgICAgICAgc2VsZi5kZXZpY2UgPSBkZXZpY2Ugb3IgdG9yY2guZGV2aWNlKCJjdWRhIiBpZiB0b3JjaC5jdWRhLmlzX2F2YWlsYWJsZSgpIGVsc2UgImNwdSIpCiAgICAgICAgc2VsZi5lbWJlZGRlciA9IFNlbnRlbmNlVHJhbnNmb3JtZXIoZW1iZWRkZXJfbmFtZSkKICAgICAgICBzZWxmLm1vZGVsX2xvYWRlZCA9IEZhbHNlCiAgICAgICAgCiAgICAgICAgc2VsZi5sYiA9IExhYmVsQmluYXJpemVyKCkKICAgICAgICBzZWxmLmxiLmZpdChbIndoYXQiLCAid2hvIiwgImhvd19tYW55IiwgIndoaWNoIiwgIndoeSIsICJ3aGVuIiwgIm90aGVyIl0pCiAgICAKICAgIGRlZiBfcmVzb2x2ZV9tb2RlbF9wYXRoKHNlbGYsIG1vZGVsX3BhdGgsIGZpbGVuYW1lKToKICAgICAgICAiIiJSZXNvbHZlIG1vZGVsIHBhdGggZnJvbSBtdWx0aXBsZSBwb3NzaWJsZSBsb2NhdGlvbnMuIiIiCiAgICAgICAgcG9zc2libGVfcGF0aHMgPSBbCiAgICAgICAgICAgIG1vZGVsX3BhdGgsCiAgICAgICAgICAgIG9zLnBhdGguam9pbihvcy5nZXRjd2QoKSwgbW9kZWxfcGF0aCksCiAgICAgICAgICAgIG9zLnBhdGguam9pbihvcy5wYXRoLmRpcm5hbWUob3MucGF0aC5hYnNwYXRoKF9fZmlsZV9fKSksICdtb2RlbHMnLCBmaWxlbmFtZSksCiAgICAgICAgXQogICAgICAgIAogICAgICAgICMgTm90ZWJvb2sgY29udGV4dCBmYWxsYmFjawogICAgICAgIG5vdGVib29rX2RpciA9IG9zLmdldGN3ZCgpCiAgICAgICAgaWYgb3MucGF0aC5iYXNlbmFtZShub3RlYm9va19kaXIpID09ICdub3RlYm9va3MnOgogICAgICAgICAgICBwb3NzaWJsZV9wYXRocy5hcHBlbmQob3MucGF0aC5qb2luKG9zLnBhdGguZGlybmFtZShub3RlYm9va19kaXIpLCBtb2RlbF9wYXRoKSkKICAgICAgICAKICAgICAgICBmb3IgcCBpbiBwb3NzaWJsZV9wYXRoczoKICAgICAgICAgICAgaWYgb3MucGF0aC5leGlzdHMocCk6CiAgICAgICAgICAgICAgICByZXR1cm4gcCwgcG9zc2libGVfcGF0aHMKICAgICAgICByZXR1cm4gTm9uZSwgcG9zc2libGVfcGF0aHMKICAgIAogICAgZGVmIGV4dHJhY3RfcXVlc3Rpb25fdHlwZShzZWxmLCBxcyk6CiAgICAgICAgIiIiQ2xhc3NpZnkgcXVlc3Rpb24gdHlwZSBhbmQgcmV0dXJuIG9uZS1ob3QgZW5jb2RpbmcuIiIiCiAgICAgICAgdHlwZXMgPSBbXQogICAgICAgIGZvciBxIGluIHFzOgogICAgICAgICAgICBxID0gcS5zdHJpcCgpCiAgICAgICAgICAgIGlmIHEuc3RhcnRzd2l0aCgi5LuA5LmIIik6CiAgICAgICAgICAgICAgICB0eXBlcy5hcHBlbmQoIndoYXQiKQogICAgICAgICAgICBlbGlmIHEuc3RhcnRzd2l0aCgi6LCBIik6CiAgICAgICAgICAgICAgICB0eXBlcy5hcHBlbmQoIndobyIpCiAgICAgICAgICAgIGVsaWYgcS5zdGFydHN3aXRoKCLlpJrlsJEiKSBvciBxLnN0YXJ0c3dpdGgoIuWHoCIpOgogICAgICAgICAgICAgICAgdHlwZXMuYXBwZW5kKCJob3dfbWFueSIpCiAgICAgICAgICAgIGVsaWYgcS5zdGFydHN3aXRoKCLlk6oiKToKICAgICAgICAgICAgICAgIHR5cGVzLmFwcGVuZCgid2hpY2giKQogICAgICAgICAgICBlbGlmIHEuc3RhcnRzd2l0aCgi5Li65L2VIikgb3IgcS5zdGFydHN3aXRoKCLkuLrku4DkuYgiKToKICAgICAgICAgICAgICAgIHR5cGVzLmFwcGVuZCgid2h5IikKICAgICAgICAgICAgZWxpZiBxLnN0YXJ0c3dpdGgoIuS9leaXtiIpIG9yIHEuc3RhcnRzd2l0aCgi5LuA5LmI5pe25YCZIik6CiAgICAgICAgICAgICAgICB0eXBlcy5hcHBlbmQoIndoZW4iKQogICAgICAgICAgICBlbHNlOgogICAgICAgICAgICAgICAgdHlwZXMuYXBwZW5kKCJvdGhlciIpCiAgICAgICAgcmV0dXJuIHNlbGYubGIudHJhbnNmb3JtKHR5cGVzKQogICAgCiAgICBkZWYgX2V4dHJhY3RfZmVhdHVyZXMoc2VsZiwgcXVlc3Rpb25zLCBjaHVua3MpOgogICAgICAgICIiIkV4dHJhY3QgZmVhdHVyZSBtYXRyaXggZnJvbSAocXVlc3Rpb24sIGNodW5rKSBwYWlycy4KICAgICAgICAKICAgICAgICBSZXR1cm5zOgogICAgICAgICAgICBucC5uZGFycmF5OiBGZWF0dXJlIG1hdHJpeCBvZiBzaGFwZSAoTiwgZmVhdHVyZV9kaW0pLgogICAgICAgICIiIgogICAgICAgIGlmIGxlbihxdWVzdGlvbnMpICE9IGxlbihjaHVua3MpOgogICAgICAgICAgICByYWlzZSBWYWx1ZUVycm9yKCJOdW1iZXIgb2YgcXVlc3Rpb25zIGFuZCBjaHVua3MgbXVzdCBtYXRjaC4iKQogICAgICAgIAogICAgICAgICMgMS4gRW1iZWRkaW5ncwogICAgICAgIHNfZW1icyA9IHNlbGYuZW1iZWRkZXIuZW5jb2RlKGNodW5rcywgY29udmVydF90b190ZW5zb3I9VHJ1ZSwgZGV2aWNlPXNlbGYuZGV2aWNlKS5jcHUoKS5udW1weSgpCiAgICAgICAgcV9lbWJzID0gc2VsZi5lbWJlZGRlci5lbmNvZGUocXVlc3Rpb25zLCBjb252ZXJ0X3RvX3RlbnNvcj1UcnVlLCBkZXZpY2U9c2VsZi5kZXZpY2UpLmNwdSgpLm51bXB5KCkKICAgICAgICAKICAgICAgICAjIDIuIFN0cmluZyBMZW5ndGhzCiAgICAgICAgcV9sZW5zID0gbnAuYXJyYXkoW2xlbihxKSBmb3IgcSBpbiBxdWVzdGlvbnNdKS5yZXNoYXBlKC0xLCAxKQogICAgICAgICMgRHVtbXkgZ29sZCBsZW5ndGggKG5vdCBhdmFpbGFibGUgYXQgaW5mZXJlbmNlIHRpbWUpCiAgICAgICAgZ29sZF9sZW5zID0gbnAub25lcygobGVuKHF1ZXN0aW9ucyksIDEpKSAqIDEwCiAgICAgICAgCiAgICAgICAgIyAzLiBURi1JREYgLyBCTTI1CiAgICAgICAgdHJ5OgogICAgICAgICAgICB2ZWN0b3JpemVyID0gVGZpZGZWZWN0b3JpemVyKCkuZml0KGNodW5rcyArIHF1ZXN0aW9ucykKICAgICAgICAgICAgdGV4dF90ZmlkZiA9IHZlY3Rvcml6ZXIudHJhbnNmb3JtKGNodW5rcykKICAgICAgICAgICAgcV90ZmlkZiA9IHZlY3Rvcml6ZXIudHJhbnNmb3JtKHF1ZXN0aW9ucykKICAgICAgICAgICAgCiAgICAgICAgICAgIHNjb3JlcyA9IChxX3RmaWRmICogdGV4dF90ZmlkZi5UKS50b2FycmF5KCkKICAgICAgICAgICAgYm0yNV9kaWFnID0gbnAuZGlhZyhzY29yZXMpLnJlc2hhcGUoLTEsIDEpCiAgICAgICAgICAgIGJtMjVfcmFuayA9IG5wLmFyZ3NvcnQoLXNjb3JlcywgYXhpcz0xKVs6LCAwXS5yZXNoYXBlKC0xLCAxKQogICAgICAgICAgICAKICAgICAgICAgICAgc2ltcyA9IGNvc2luZV9zaW1pbGFyaXR5KHFfdGZpZGYsIHRleHRfdGZpZGYpCiAgICAgICAgICAgIHRmaWRmX2NvcyA9IG5wLmRpYWcoc2ltcykucmVzaGFwZSgtMSwgMSkKICAgICAgICBleGNlcHQgVmFsdWVFcnJvcjoKICAgICAgICAgICAgbiA9IGxlbihxdWVzdGlvbnMpCiAgICAgICAgICAgIGJtMjVfZGlhZyA9IG5wLnplcm9zKChuLCAxKSkKICAgICAgICAgICAgYm0yNV9yYW5rID0gbnAuemVyb3MoKG4sIDEpKQogICAgICAgICAgICB0ZmlkZl9jb3MgPSBucC56ZXJvcygobiwgMSkpCiAgICAgICAgCiAgICAgICAgIyA0LiBOZXVyYWwgRW1iZWRkaW5nIGRpc3RhbmNlcwogICAgICAgIGRlZiBzYWZlX25vcm0odik6CiAgICAgICAgICAgIHJldHVybiB2IC8gKG5wLmxpbmFsZy5ub3JtKHYsIGF4aXM9MSwga2VlcGRpbXM9VHJ1ZSkgKyAxZS04KQogICAgICAgIAogICAgICAgIGNvc19zaW0gPSBucC5zdW0oc2FmZV9ub3JtKHFfZW1icykgKiBzYWZlX25vcm0oc19lbWJzKSwgYXhpcz0xLCBrZWVwZGltcz1UcnVlKQogICAgICAgIGV1X2Rpc3QgPSBucC5saW5hbGcubm9ybShxX2VtYnMgLSBzX2VtYnMsIGF4aXM9MSwga2VlcGRpbXM9VHJ1ZSkKICAgICAgICBtYW5fZGlzdCA9IG5wLnN1bShucC5hYnMocV9lbWJzIC0gc19lbWJzKSwgYXhpcz0xLCBrZWVwZGltcz1UcnVlKQogICAgICAgIAogICAgICAgICMgSmFjY2FyZAogICAgICAgIGRlZiBqYWNjYXJkKGEsIGIpOgogICAgICAgICAgICBzYSwgc2IgPSBzZXQoYS5zcGxpdCgpKSwgc2V0KGIuc3BsaXQoKSkKICAgICAgICAgICAgaWYgbm90IHNhIG9yIG5vdCBzYjoKICAgICAgICAgICAgICAgIHJldHVybiAwLjAKICAgICAgICAgICAgcmV0dXJuIGxlbihzYSAmIHNiKSAvIGxlbihzYSB8IHNiKQogICAgICAgIGphY2NhcmRfc2ltID0gbnAuYXJyYXkoW2phY2NhcmQocSwgcykgZm9yIHEsIHMgaW4gemlwKHF1ZXN0aW9ucywgY2h1bmtzKV0pLnJlc2hhcGUoLTEsIDEpCiAgICAgICAgCiAgICAgICAgIyA1LiBRdWVzdGlvbiBUeXBlCiAgICAgICAgcV90eXBlX29oID0gc2VsZi5leHRyYWN0X3F1ZXN0aW9uX3R5cGUocXVlc3Rpb25zKQogICAgICAgIAogICAgICAgICMgQ29tYmluZSBudW1lcmljYWwgZmVhdHVyZXMKICAgICAgICBudW1fZmVhdHMgPSBucC5jb25jYXRlbmF0ZShbCiAgICAgICAgICAgIHFfbGVucywgZ29sZF9sZW5zLCBibTI1X2RpYWcsIGJtMjVfcmFuaywgY29zX3NpbSwgZXVfZGlzdCwgbWFuX2Rpc3QsIGphY2NhcmRfc2ltLCB0ZmlkZl9jb3MKICAgICAgICBdLCBheGlzPTEpCiAgICAgICAgCiAgICAgICAgIyBTdGFuZGFyZGl6ZSBudW1lcmljYWwgZmVhdHVyZXMKICAgICAgICBtZWFuID0gbnAubWVhbihudW1fZmVhdHMsIGF4aXM9MCkKICAgICAgICBzdGQgPSBucC5zdGQobnVtX2ZlYXRzLCBheGlzPTApICsgMWUtOAogICAgICAgIG51bV9mZWF0cyA9IChudW1fZmVhdHMgLSBtZWFuKSAvIHN0ZAogICAgICAgIAogICAgICAgIGZlYXR1cmVzID0gbnAuY29uY2F0ZW5hdGUoW3NfZW1icywgbnVtX2ZlYXRzLCBxX3R5cGVfb2hdLCBheGlzPTEpCiAgICAgICAgcmV0dXJuIGZlYXR1cmVzCiAgICAKICAgIGRlZiBwcmVkaWN0X2JhdGNoKHNlbGYsIHF1ZXN0aW9ucywgY2h1bmtzKToKICAgICAgICAiIiJQcmVkaWN0IGNvbmZpZGVuY2Ugc2NvcmVzLiBNdXN0IGJlIGltcGxlbWVudGVkIGJ5IHN1YmNsYXNzLiIiIgogICAgICAgIHJhaXNlIE5vdEltcGxlbWVudGVkRXJyb3IKICAgIAogICAgZGVmIHByZWRpY3Rfc2luZ2xlKHNlbGYsIHF1ZXN0aW9uLCBjaHVuayk6CiAgICAgICAgIiIiUHJlZGljdCBjb25maWRlbmNlIHNjb3JlIGZvciBhIHNpbmdsZSAocXVlc3Rpb24sIGNodW5rKSBwYWlyLiIiIgogICAgICAgIHJldHVybiBzZWxmLnByZWRpY3RfYmF0Y2goW3F1ZXN0aW9uXSwgW2NodW5rXSlbMF0KCgpjbGFzcyBSQUdDb25maWRlbmNlU2NvcmVyKEJhc2VDb25maWRlbmNlU2NvcmVyKToKICAgICIiIk1MUC1iYXNlZCBSQUcgY29uZmlkZW5jZSBzY29yZXIgdXNpbmcgdHJhaW5fdGVzdF9tb2RlbC5wdC4iIiIKICAgIAogICAgZGVmIF9faW5pdF9fKHNlbGYsIG1vZGVsX3BhdGg9ImlzX1JBR19hYmxlL21vZGVscy90cmFpbl90ZXN0X21vZGVsLnB0IiwKICAgICAgICAgICAgICAgICBlbWJlZGRlcl9uYW1lPSJhbGwtTWluaUxNLUw2LXYyIiwgZGV2aWNlPU5vbmUpOgogICAgICAgIHN1cGVyKCkuX19pbml0X18oZW1iZWRkZXJfbmFtZSwgZGV2aWNlKQogICAgICAgIAogICAgICAgIGZpbmFsX3BhdGgsIHBvc3NpYmxlX3BhdGhzID0gc2VsZi5fcmVzb2x2ZV9tb2RlbF9wYXRoKG1vZGVsX3BhdGgsICd0cmFpbl90ZXN0X21vZGVsLnB0JykKICAgICAgICAKICAgICAgICAjIERldGVybWluZSBpbnB1dF9kaW0gZnJvbSBtZXRhIG9yIHN0YXRlX2RpY3QKICAgICAgICBpbnB1dF9kaW0gPSBJTlBVVF9ESU0KICAgICAgICBzdGF0ZV9kaWN0X2NhY2hlID0gTm9uZQogICAgICAgIAogICAgICAgIGlmIGZpbmFsX3BhdGg6CiAgICAgICAgICAgIG1ldGFfcGF0aCA9IGZpbmFsX3BhdGggKyAiLm1ldGEuanNvbiIKICAgICAgICAgICAgaWYgb3MucGF0aC5leGlzdHMobWV0YV9wYXRoKToKICAgICAgICAgICAgICAgIHdpdGggb3BlbihtZXRhX3BhdGgsICJyIiwgZW5jb2Rpbmc9InV0Zi04IikgYXMgZjoKICAgICAgICAgICAgICAgICAgICBtZXRhID0ganNvbi5sb2FkKGYpCiAgICAgICAgICAgICAgICAgICAgaW5wdXRfZGltID0gbWV0YS5nZXQoImlucHV0X2RpbSIsIElOUFVUX0RJTSkKICAgICAgICAgICAgZWxzZToKICAgICAgICAgICAgICAgIHN0YXRlX2RpY3RfY2FjaGUgPSB0b3JjaC5sb2FkKGZpbmFsX3BhdGgsIG1hcF9sb2NhdGlvbj1zZWxmLmRldmljZSkKICAgICAgICAgICAgICAgIGlmICdmYzEud2VpZ2h0JyBpbiBzdGF0ZV9kaWN0X2NhY2hlOgogICAgICAgICAgICAgICAgICAgIGlucHV0X2RpbSA9IHN0YXRlX2RpY3RfY2FjaGVbJ2ZjMS53ZWlnaHQnXS5zaGFwZVsxXQogICAgICAgIAogICAgICAgIHNlbGYubW9kZWwgPSBNTFBDbGFzc2lmaWVyKGlucHV0X2RpbT1pbnB1dF9kaW0pCiAgICAgICAgCiAgICAgICAgaWYgZmluYWxfcGF0aDoKICAgICAgICAgICAgaWYgc3RhdGVfZGljdF9jYWNoZSBpcyBOb25lOgogICAgICAgICAgICAgICAgc3RhdGVfZGljdF9jYWNoZSA9IHRvcmNoLmxvYWQoZmluYWxfcGF0aCwgbWFwX2xvY2F0aW9uPXNlbGYuZGV2aWNlKQogICAgICAgICAgICBzZWxmLm1vZGVsLmxvYWRfc3RhdGVfZGljdChzdGF0ZV9kaWN0X2NhY2hlKQogICAgICAgICAgICBzZWxmLm1vZGVsLnRvKHNlbGYuZGV2aWNlKQogICAgICAgICAgICBzZWxmLm1vZGVsLmV2YWwoKQogICAgICAgICAgICBzZWxmLm1vZGVsX2xvYWRlZCA9IFRydWUKICAgICAgICAgICAgcHJpbnQoZiLinIUgUkFHQ29uZmlkZW5jZVNjb3JlciBsb2FkZWQgZnJvbSB7ZmluYWxfcGF0aH0gKGlucHV0X2RpbT17aW5wdXRfZGltfSkiKQogICAgICAgIGVsc2U6CiAgICAgICAgICAgIHByaW50KGYi4pqg77iPIFdhcm5pbmc6IFJBRyBtb2RlbCBub3QgZm91bmQuIENoZWNrZWQgcGF0aHM6IHtwb3NzaWJsZV9wYXRoc30uIFdpbGwgcmV0dXJuIGRlZmF1bHQgc2NvcmVzLiIpCiAgICAKICAgIEB0b3JjaC5ub19ncmFkKCkKICAgIGRlZiBwcmVkaWN0X2JhdGNoKHNlbGYsIHF1ZXN0aW9ucywgY2h1bmtzKToKICAgICAgICAiIiJQcmVkaWN0IGNvbmZpZGVuY2Ugc2NvcmVzIHVzaW5nIE1MUCBtb2RlbC4KICAgICAgICAKICAgICAgICBSZXR1cm5zOgogICAgICAgICAgICBucC5uZGFycmF5OiBBcnJheSBvZiBwcm9iYWJpbGl0aWVzIGluIFswLCAxXS4KICAgICAgICAiIiIKICAgICAgICBpZiBub3Qgc2VsZi5tb2RlbF9sb2FkZWQ6CiAgICAgICAgICAgIHJldHVybiBucC5vbmVzKGxlbihxdWVzdGlvbnMpKSAqIDAuNQogICAgICAgIAogICAgICAgIGZlYXR1cmVzID0gc2VsZi5fZXh0cmFjdF9mZWF0dXJlcyhxdWVzdGlvbnMsIGNodW5rcykKICAgICAgICAKICAgICAgICAjIEFsaWduIGRpbWVuc2lvbiB0byBtb2RlbCdzIGV4cGVjdGVkIGlucHV0CiAgICAgICAgaW5wdXRfZGltID0gc2VsZi5tb2RlbC5mYzEuaW5fZmVhdHVyZXMKICAgICAgICBpZiBmZWF0dXJlcy5zaGFwZVsxXSA8IGlucHV0X2RpbToKICAgICAgICAgICAgcGFkID0gbnAuemVyb3MoKGZlYXR1cmVzLnNoYXBlWzBdLCBpbnB1dF9kaW0gLSBmZWF0dXJlcy5zaGFwZVsxXSkpCiAgICAgICAgICAgIGZlYXR1cmVzID0gbnAuY29uY2F0ZW5hdGUoW2ZlYXR1cmVzLCBwYWRdLCBheGlzPTEpCiAgICAgICAgZWxpZiBmZWF0dXJlcy5zaGFwZVsxXSA+IGlucHV0X2RpbToKICAgICAgICAgICAgZmVhdHVyZXMgPSBmZWF0dXJlc1s6LCA6aW5wdXRfZGltXQogICAgICAgIAogICAgICAgIGZlYXR1cmVzX3QgPSB0b3JjaC50ZW5zb3IoZmVhdHVyZXMsIGR0eXBlPXRvcmNoLmZsb2F0MzIsIGRldmljZT1zZWxmLmRldmljZSkKICAgICAgICBsb2dpdHMgPSBzZWxmLm1vZGVsKGZlYXR1cmVzX3QpCiAgICAgICAgcHJvYnMgPSB0b3JjaC5zaWdtb2lkKGxvZ2l0cykKICAgICAgICAKICAgICAgICByZXR1cm4gcHJvYnMuY3B1KCkubnVtcHkoKQoKCmNsYXNzIFhHQkNvbmZpZGVuY2VTY29yZXIoQmFzZUNvbmZpZGVuY2VTY29yZXIpOgogICAgIiIiWEdCb29zdC1iYXNlZCBSQUcgY29uZmlkZW5jZSBzY29yZXIgdXNpbmcgeGdiX21vZGVsLnBrbC4iIiIKICAgIAogICAgZGVmIF9faW5pdF9fKHNlbGYsIG1vZGVsX3BhdGg9ImlzX1JBR19hYmxlL21vZGVscy94Z2JfbW9kZWwucGtsIiwKICAgICAgICAgICAgICAgICBlbWJlZGRlcl9uYW1lPSJhbGwtTWluaUxNLUw2LXYyIiwgZGV2aWNlPU5vbmUpOgogICAgICAgIHN1cGVyKCkuX19pbml0X18oZW1iZWRkZXJfbmFtZSwgZGV2aWNlKQogICAgICAgIGltcG9ydCBqb2JsaWIKICAgICAgICAKICAgICAgICBmaW5hbF9wYXRoLCBwb3NzaWJsZV9wYXRocyA9IHNlbGYuX3Jlc29sdmVfbW9kZWxfcGF0aChtb2RlbF9wYXRoLCAneGdiX21vZGVsLnBrbCcpCiAgICAgICAgCiAgICAgICAgaWYgZmluYWxfcGF0aDoKICAgICAgICAgICAgc2VsZi5tb2RlbCA9IGpvYmxpYi5sb2FkKGZpbmFsX3BhdGgpCiAgICAgICAgICAgIHNlbGYubW9kZWxfbG9hZGVkID0gVHJ1ZQogICAgICAgICAgICBwcmludChmIuKchSBYR0JDb25maWRlbmNlU2NvcmVyIGxvYWRlZCBmcm9tIHtmaW5hbF9wYXRofSIpCiAgICAgICAgZWxzZToKICAgICAgICAgICAgcHJpbnQoZiLimqDvuI8gV2FybmluZzogWEdCIG1vZGVsIG5vdCBmb3VuZC4gQ2hlY2tlZCBwYXRoczoge3Bvc3NpYmxlX3BhdGhzfSIpCiAgICAgICAgICAgIHNlbGYubW9kZWwgPSBOb25lCiAgICAKICAgIGRlZiBwcmVkaWN0X2JhdGNoKHNlbGYsIHF1ZXN0aW9ucywgY2h1bmtzKToKICAgICAgICAiIiJQcmVkaWN0IGNvbmZpZGVuY2Ugc2NvcmVzIHVzaW5nIFhHQm9vc3QgbW9kZWwuCiAgICAgICAgCiAgICAgICAgUmV0dXJuczoKICAgICAgICAgICAgbnAubmRhcnJheTogQXJyYXkgb2YgcHJvYmFiaWxpdGllcyBpbiBbMCwgMV0gZm9yICJSZXRyaWV2YWJsZSIgY2xhc3MuCiAgICAgICAgIiIiCiAgICAgICAgaWYgbm90IHNlbGYubW9kZWxfbG9hZGVkOgogICAgICAgICAgICByZXR1cm4gbnAub25lcyhsZW4ocXVlc3Rpb25zKSkgKiAwLjUKICAgICAgICAKICAgICAgICBmZWF0dXJlcyA9IHNlbGYuX2V4dHJhY3RfZmVhdHVyZXMocXVlc3Rpb25zLCBjaHVua3MpCiAgICAgICAgCiAgICAgICAgIyBBbGlnbiBmZWF0dXJlIGRpbWVuc2lvbiB0byBtYXRjaCB0cmFpbmVkIG1vZGVsCiAgICAgICAgZXhwZWN0ZWRfZGltID0gc2VsZi5tb2RlbC5uX2ZlYXR1cmVzX2luXwogICAgICAgIGlmIGZlYXR1cmVzLnNoYXBlWzFdIDwgZXhwZWN0ZWRfZGltOgogICAgICAgICAgICBwYWQgPSBucC56ZXJvcygoZmVhdHVyZXMuc2hhcGVbMF0sIGV4cGVjdGVkX2RpbSAtIGZlYXR1cmVzLnNoYXBlWzFdKSkKICAgICAgICAgICAgZmVhdHVyZXMgPSBucC5jb25jYXRlbmF0ZShbZmVhdHVyZXMsIHBhZF0sIGF4aXM9MSkKICAgICAgICBlbGlmIGZlYXR1cmVzLnNoYXBlWzFdID4gZXhwZWN0ZWRfZGltOgogICAgICAgICAgICBmZWF0dXJlcyA9IGZlYXR1cmVzWzosIDpleHBlY3RlZF9kaW1dCiAgICAgICAgCiAgICAgICAgIyBYR0Jvb3N0IHByZWRpY3RfcHJvYmEg4oaSIHByb2JhYmlsaXR5IG9mICJSZXRyaWV2YWJsZSIgKGNsYXNzIDEpCiAgICAgICAgcHJvYnMgPSBzZWxmLm1vZGVsLnByZWRpY3RfcHJvYmEoZmVhdHVyZXMpWzosIDFdCiAgICAgICAgCiAgICAgICAgcmV0dXJuIHByb2JzCg=='
    ,'is_RAG_able/__init__.py': ''
}

print(f'Setting up {len(files)} core files...')
for path, content in files.items():
    write_file(path, content)

print('\n✅ Core files ready!')


## 2.1 Setup XGBoost Scorer Model
The XGBoost scorer model is used by AdaptiveHybridRAG for per-chunk soft routing.
It predicts how likely a (question, chunk) pair is retrievable by traditional RAG.

In [ ]:
import os, base64

model_dir = 'is_RAG_able/models'
model_path = os.path.join(model_dir, 'xgb_model.pkl')
os.makedirs(model_dir, exist_ok=True)

xgb_b64 = '...'
with open(model_path, 'wb') as f:
    f.write(base64.b64decode(xgb_b64))
size_mb = os.path.getsize(model_path) / (1024 * 1024)
print(f'✅ XGBoost model written: {model_path} ({size_mb:.1f} MB)')


## 2.5 Setup SQuAD Dataset (Optional)

This cell will automatically download and generate the SQuAD dataset.
**Skip this cell if you only want to use Sample Essays mode.**

In [ ]:
import os
import base64

# Check if SQuAD data already exists
if os.path.exists('data/processed_squad.py'):
    size_mb = os.path.getsize('data/processed_squad.py') / (1024 * 1024)
    print(f'✅ SQuAD dataset already exists ({size_mb:.1f} MB)')
else:
    print('📥 Downloading SQuAD dataset...')
    !wget -q --show-progress https://rajpurkar.github.io/SQuAD-explorer/dataset/train-v2.0.json -O data/train-v2.0.json
    
    # Write the processing script
    script_b64 = 'IiIiClNjcmlwdCB0byBjbGVhbiBhbmQgcHJvY2VzcyBTUXVBRCBkYXRhc2V0IChKU09OIG9yIENTViBmb3JtYXQpLgoKVGhpcyBzY3JpcHQ6CjEuIEZpbHRlcnMgb3V0IHVuaXF1ZSB0aXRsZXMKMi4gQ29tYmluZXMgZHVwbGljYXRlIGNvbnRleHRzIGJhc2VkIG9uIHRpdGxlCjMuIEV4dHJhY3RzIHF1ZXN0aW9uIGFuZCBhbnN3ZXIgcGFpcnMgZm9yIGVhY2ggY29udGV4dAoKT3V0cHV0IGZvcm1hdCBtYXRjaGVzIHNhbXBsZV9lc3NheXMucHkgc3RydWN0dXJlLgoKU3VwcG9ydHM6Ci0gU1F1QUQgdjIuMCBKU09OIGZvcm1hdCAodHJhaW4tdjIuMC5qc29uKQotIENTViBmb3JtYXQgKHRyYWluLmNzdikKIiIiCgppbXBvcnQgY3N2CmltcG9ydCBhc3QKaW1wb3J0IHJlCmltcG9ydCBzeXMKaW1wb3J0IGlvCmltcG9ydCBqc29uCmltcG9ydCBvcwoKIyBGaXggV2luZG93cyBjb25zb2xlIGVuY29kaW5nCnN5cy5zdGRvdXQgPSBpby5UZXh0SU9XcmFwcGVyKHN5cy5zdGRvdXQuYnVmZmVyLCBlbmNvZGluZz0ndXRmLTgnKQoKCmRlZiBsb2FkX2Nzdl9kYXRhKGZpbGVfcGF0aDogc3RyKSAtPiBsaXN0OgogICAgIiIiCiAgICBMb2FkIENTViBkYXRhc2V0LgogICAgCiAgICBBcmdzOgogICAgICAgIGZpbGVfcGF0aDogUGF0aCB0byB0aGUgQ1NWIGZpbGUKICAgICAgICAKICAgIFJldHVybnM6CiAgICAgICAgTGlzdCBvZiByb3cgZGljdGlvbmFyaWVzCiAgICAiIiIKICAgIGRhdGEgPSBbXQogICAgd2l0aCBvcGVuKGZpbGVfcGF0aCwgJ3InLCBlbmNvZGluZz0ndXRmLTgnKSBhcyBmOgogICAgICAgIHJlYWRlciA9IGNzdi5EaWN0UmVhZGVyKGYpCiAgICAgICAgZm9yIHJvdyBpbiByZWFkZXI6CiAgICAgICAgICAgIGRhdGEuYXBwZW5kKHJvdykKICAgIHByaW50KGYiTG9hZGVkIHtsZW4oZGF0YSl9IHJvd3MgZnJvbSBDU1YiKQogICAgcmV0dXJuIGRhdGEKCgpkZWYgbG9hZF9qc29uX2RhdGEoZmlsZV9wYXRoOiBzdHIpIC0+IGxpc3Q6CiAgICAiIiIKICAgIExvYWQgU1F1QUQgdjIuMCBKU09OIGRhdGFzZXQgYW5kIGNvbnZlcnQgdG8gcm93IGZvcm1hdC4KICAgIAogICAgQXJnczoKICAgICAgICBmaWxlX3BhdGg6IFBhdGggdG8gdGhlIEpTT04gZmlsZSAodHJhaW4tdjIuMC5qc29uKQogICAgICAgIAogICAgUmV0dXJuczoKICAgICAgICBMaXN0IG9mIHJvdyBkaWN0aW9uYXJpZXMgd2l0aCBrZXlzOiB0aXRsZSwgY29udGV4dCwgcXVlc3Rpb24sIGFuc3dlcnMKICAgICIiIgogICAgd2l0aCBvcGVuKGZpbGVfcGF0aCwgJ3InLCBlbmNvZGluZz0ndXRmLTgnKSBhcyBmOgogICAgICAgIHNxdWFkX2RhdGEgPSBqc29uLmxvYWQoZikKICAgIAogICAgZGF0YSA9IFtdCiAgICBmb3IgYXJ0aWNsZSBpbiBzcXVhZF9kYXRhWydkYXRhJ106CiAgICAgICAgdGl0bGUgPSBhcnRpY2xlWyd0aXRsZSddCiAgICAgICAgZm9yIHBhcmFncmFwaCBpbiBhcnRpY2xlWydwYXJhZ3JhcGhzJ106CiAgICAgICAgICAgIGNvbnRleHQgPSBwYXJhZ3JhcGhbJ2NvbnRleHQnXQogICAgICAgICAgICBmb3IgcWEgaW4gcGFyYWdyYXBoWydxYXMnXToKICAgICAgICAgICAgICAgIHF1ZXN0aW9uID0gcWFbJ3F1ZXN0aW9uJ10KICAgICAgICAgICAgICAgICMgSGFuZGxlIGJvdGggdjEgYW5kIHYyIGZvcm1hdAogICAgICAgICAgICAgICAgYW5zd2VycyA9IHFhLmdldCgnYW5zd2VycycsIFtdKQogICAgICAgICAgICAgICAgaWYgbm90IGFuc3dlcnMgYW5kICdwbGF1c2libGVfYW5zd2VycycgaW4gcWE6CiAgICAgICAgICAgICAgICAgICAgYW5zd2VycyA9IHFhWydwbGF1c2libGVfYW5zd2VycyddCiAgICAgICAgICAgICAgICAKICAgICAgICAgICAgICAgICMgQ29udmVydCB0byBmb3JtYXQgZXhwZWN0ZWQgYnkgcHJvY2Vzc19kYXRhc2V0CiAgICAgICAgICAgICAgICBhbnN3ZXJfdGV4dCA9IGFuc3dlcnNbMF1bJ3RleHQnXSBpZiBhbnN3ZXJzIGVsc2UgJycKICAgICAgICAgICAgICAgIGRhdGEuYXBwZW5kKHsKICAgICAgICAgICAgICAgICAgICAndGl0bGUnOiB0aXRsZSwKICAgICAgICAgICAgICAgICAgICAnY29udGV4dCc6IGNvbnRleHQsCiAgICAgICAgICAgICAgICAgICAgJ3F1ZXN0aW9uJzogcXVlc3Rpb24sCiAgICAgICAgICAgICAgICAgICAgJ2Fuc3dlcnMnOiBhbnN3ZXJfdGV4dCwgICMgQWxyZWFkeSBwYXJzZWQKICAgICAgICAgICAgICAgICAgICAnX3BhcnNlZCc6IFRydWUgICMgRmxhZyB0byBza2lwIHBhcnNlX2Fuc3dlcgogICAgICAgICAgICAgICAgfSkKICAgIAogICAgcHJpbnQoZiJMb2FkZWQge2xlbihkYXRhKX0gUUEgcGFpcnMgZnJvbSBKU09OIikKICAgIHJldHVybiBkYXRhCgoKZGVmIHBhcnNlX2Fuc3dlcihhbnN3ZXJzX3N0cjogc3RyKSAtPiBzdHI6CiAgICAiIiIKICAgIFBhcnNlIHRoZSBhbnN3ZXIgc3RyaW5nIGZyb20gQ1NWIGZvcm1hdC4KICAgIAogICAgRm9ybWF0OiB7J3RleHQnOiBhcnJheShbJ2Fuc3dlciB0ZXh0J10sIGR0eXBlPW9iamVjdCksICdhbnN3ZXJfc3RhcnQnOiBhcnJheShbMTIzXSwgZHR5cGU9aW50MzIpfQogICAgCiAgICBBcmdzOgogICAgICAgIGFuc3dlcnNfc3RyOiBSYXcgYW5zd2VyIHN0cmluZyBmcm9tIENTVgogICAgICAgIAogICAgUmV0dXJuczoKICAgICAgICBFeHRyYWN0ZWQgYW5zd2VyIHRleHQsIG9yIGVtcHR5IHN0cmluZyBpZiBwYXJzaW5nIGZhaWxzCiAgICAiIiIKICAgIHRyeToKICAgICAgICAjIENsZWFuIG51bXB5IGFycmF5IGZvcm1hdCB0byBQeXRob24gbGlzdCBmb3JtYXQKICAgICAgICBjbGVhbmVkID0gYW5zd2Vyc19zdHIucmVwbGFjZSgiYXJyYXkoIiwgIiIpLnJlcGxhY2UoIiwgZHR5cGU9b2JqZWN0KSIsICIiKS5yZXBsYWNlKCIsIGR0eXBlPWludDMyKSIsICIiKQogICAgICAgIGFuc3dlcnNfZGljdCA9IGFzdC5saXRlcmFsX2V2YWwoY2xlYW5lZCkKICAgICAgICAKICAgICAgICB0ZXh0X2xpc3QgPSBhbnN3ZXJzX2RpY3QuZ2V0KCd0ZXh0JywgW10pCiAgICAgICAgaWYgdGV4dF9saXN0IGFuZCBsZW4odGV4dF9saXN0KSA+IDA6CiAgICAgICAgICAgIHJldHVybiB0ZXh0X2xpc3RbMF0uc3RyaXAoKQogICAgZXhjZXB0IEV4Y2VwdGlvbjoKICAgICAgICBwYXNzCiAgICByZXR1cm4gIiIKCgpkZWYgcHJvY2Vzc19kYXRhc2V0KGRhdGE6IGxpc3QpIC0+IHR1cGxlOgogICAgIiIiCiAgICBQcm9jZXNzIHRoZSBkYXRhc2V0IHRvIGV4dHJhY3QgdW5pcXVlIHRpdGxlcywgY29tYmluZWQgY29udGV4dHMsIGFuZCBRQSBwYWlycy4KICAgIAogICAgQXJnczoKICAgICAgICBkYXRhOiBMaXN0IG9mIHJvdyBkaWN0aW9uYXJpZXMgZnJvbSBDU1Ygb3IgSlNPTgogICAgICAgIAogICAgUmV0dXJuczoKICAgICAgICBUdXBsZSBvZiAoY29udGV4dHNfZGljdCwgcXVlc3Rpb25zX2RpY3QpCiAgICAgICAgLSBjb250ZXh0c19kaWN0OiB7dGl0bGU6IGNvbWJpbmVkX2NvbnRleHR9CiAgICAgICAgLSBxdWVzdGlvbnNfZGljdDoge3RpdGxlOiBbKHF1ZXN0aW9uLCBhbnN3ZXIpLCAuLi5dfQogICAgIiIiCiAgICAjIFN0b3JlIHVuaXF1ZSBjb250ZXh0cyBwZXIgdGl0bGUgKHByZXNlcnZlIG9yZGVyLCBhdm9pZCBkdXBsaWNhdGVzKQogICAgdGl0bGVfY29udGV4dHMgPSB7fSAgIyB7dGl0bGU6IFtsaXN0IG9mIHVuaXF1ZSBjb250ZXh0IHBhcmFncmFwaHNdfQogICAgdGl0bGVfcXVlc3Rpb25zID0ge30gICMge3RpdGxlOiBbKHF1ZXN0aW9uLCBhbnN3ZXIpLCAuLi5dfQogICAgCiAgICBmb3Igcm93IGluIGRhdGE6CiAgICAgICAgdGl0bGUgPSByb3dbJ3RpdGxlJ10uc3RyaXAoKQogICAgICAgIGNvbnRleHQgPSByb3dbJ2NvbnRleHQnXS5zdHJpcCgpCiAgICAgICAgcXVlc3Rpb24gPSByb3dbJ3F1ZXN0aW9uJ10uc3RyaXAoKQogICAgICAgIAogICAgICAgICMgQ2hlY2sgaWYgYW5zd2VyIGlzIHByZS1wYXJzZWQgKGZyb20gSlNPTikgb3IgbmVlZHMgcGFyc2luZyAoZnJvbSBDU1YpCiAgICAgICAgaWYgcm93LmdldCgnX3BhcnNlZCcpOgogICAgICAgICAgICBhbnN3ZXIgPSByb3dbJ2Fuc3dlcnMnXS5zdHJpcCgpIGlmIHJvd1snYW5zd2VycyddIGVsc2UgJycKICAgICAgICBlbHNlOgogICAgICAgICAgICBhbnN3ZXIgPSBwYXJzZV9hbnN3ZXIocm93WydhbnN3ZXJzJ10pCiAgICAgICAgCiAgICAgICAgIyBJbml0aWFsaXplIHRpdGxlIGVudHJ5IGlmIG5vdCBleGlzdHMKICAgICAgICBpZiB0aXRsZSBub3QgaW4gdGl0bGVfY29udGV4dHM6CiAgICAgICAgICAgIHRpdGxlX2NvbnRleHRzW3RpdGxlXSA9IFtdCiAgICAgICAgICAgIHRpdGxlX3F1ZXN0aW9uc1t0aXRsZV0gPSBbXQogICAgICAgIAogICAgICAgICMgQWRkIGNvbnRleHQgaWYgbm90IGFscmVhZHkgcHJlc2VudCAoYXZvaWQgZHVwbGljYXRlcykKICAgICAgICBpZiBjb250ZXh0IG5vdCBpbiB0aXRsZV9jb250ZXh0c1t0aXRsZV06CiAgICAgICAgICAgIHRpdGxlX2NvbnRleHRzW3RpdGxlXS5hcHBlbmQoY29udGV4dCkKICAgICAgICAKICAgICAgICAjIEFkZCBxdWVzdGlvbi1hbnN3ZXIgcGFpciAob25seSBpZiBhbnN3ZXIgZXhpc3RzKQogICAgICAgIGlmIGFuc3dlcjoKICAgICAgICAgICAgdGl0bGVfcXVlc3Rpb25zW3RpdGxlXS5hcHBlbmQoKHF1ZXN0aW9uLCBhbnN3ZXIpKQogICAgCiAgICAjIENvbWJpbmUgY29udGV4dHMgZm9yIGVhY2ggdGl0bGUKICAgIGNvbnRleHRzX2RpY3QgPSB7fQogICAgZm9yIHRpdGxlLCBwYXJhZ3JhcGhzIGluIHRpdGxlX2NvbnRleHRzLml0ZW1zKCk6CiAgICAgICAgY29udGV4dHNfZGljdFt0aXRsZV0gPSAiXG5cbiIuam9pbihwYXJhZ3JhcGhzKQogICAgCiAgICBwcmludChmIlByb2Nlc3NlZCB7bGVuKGNvbnRleHRzX2RpY3QpfSB1bmlxdWUgdGl0bGVzIikKICAgIHByaW50KGYiVG90YWwgUUEgcGFpcnM6IHtzdW0obGVuKHFhKSBmb3IgcWEgaW4gdGl0bGVfcXVlc3Rpb25zLnZhbHVlcygpKX0iKQogICAgCiAgICByZXR1cm4gY29udGV4dHNfZGljdCwgdGl0bGVfcXVlc3Rpb25zCgoKZGVmIGdlbmVyYXRlX291dHB1dF9maWxlKGNvbnRleHRzOiBkaWN0LCBxdWVzdGlvbnM6IGRpY3QsIG91dHB1dF9wYXRoOiBzdHIpOgogICAgIiIiCiAgICBHZW5lcmF0ZSBvdXRwdXQgUHl0aG9uIGZpbGUgaW4gc2FtcGxlX2Vzc2F5cy5weSBmb3JtYXQuCiAgICAKICAgIEFyZ3M6CiAgICAgICAgY29udGV4dHM6IERpY3Rpb25hcnkgb2Yge3RpdGxlOiBjb21iaW5lZF9jb250ZXh0fQogICAgICAgIHF1ZXN0aW9uczogRGljdGlvbmFyeSBvZiB7dGl0bGU6IFsocXVlc3Rpb24sIGFuc3dlciksIC4uLl19CiAgICAgICAgb3V0cHV0X3BhdGg6IFBhdGggZm9yIG91dHB1dCBmaWxlCiAgICAiIiIKICAgIHdpdGggb3BlbihvdXRwdXRfcGF0aCwgJ3cnLCBlbmNvZGluZz0ndXRmLTgnKSBhcyBmOgogICAgICAgICMgSGVhZGVyCiAgICAgICAgZi53cml0ZSgnIiIiXG4nKQogICAgICAgIGYud3JpdGUoJ1Byb2Nlc3NlZCBTUXVBRCBEYXRhc2V0IGZvciBUaXRhblJBRyBFeHBlcmltZW50c1xuJykKICAgICAgICBmLndyaXRlKCdcbicpCiAgICAgICAgZi53cml0ZSgnVGhpcyBmaWxlIGNvbnRhaW5zIHByb2Nlc3NlZCBjb250ZXh0cyBhbmQgcXVlc3Rpb24tYW5zd2VyIHBhaXJzXG4nKQogICAgICAgIGYud3JpdGUoJ2V4dHJhY3RlZCBmcm9tIHRoZSB0cmFpbi5jc3YgZGF0YXNldC5cbicpCiAgICAgICAgZi53cml0ZSgnIiIiXG5cbicpCiAgICAgICAgCiAgICAgICAgIyBHZW5lcmF0ZSBjb250ZXh0IHZhcmlhYmxlcwogICAgICAgIGYud3JpdGUoJyMgPT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09XG4nKQogICAgICAgIGYud3JpdGUoJyMgQ09OVEVYVFMgLSBDb21iaW5lZCBwYXJhZ3JhcGhzIGJ5IHRpdGxlXG4nKQogICAgICAgIGYud3JpdGUoJyMgPT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09XG5cbicpCiAgICAgICAgCiAgICAgICAgZm9yIHRpdGxlLCBjb250ZXh0IGluIGNvbnRleHRzLml0ZW1zKCk6CiAgICAgICAgICAgICMgQ3JlYXRlIHZhbGlkIFB5dGhvbiB2YXJpYWJsZSBuYW1lCiAgICAgICAgICAgIHZhcl9uYW1lID0gY3JlYXRlX3ZhcmlhYmxlX25hbWUodGl0bGUpCiAgICAgICAgICAgICMgRXNjYXBlIGJhY2tzbGFzaGVzIGZpcnN0LCB0aGVuIHRyaXBsZSBxdW90ZXMgaW4gY29udGVudAogICAgICAgICAgICBlc2NhcGVkX2NvbnRleHQgPSBjb250ZXh0LnJlcGxhY2UoJ1xcJywgJ1xcXFwnKS5yZXBsYWNlKCciIiInLCAnXFwiXFwiXFwiJykKICAgICAgICAgICAgZi53cml0ZShmJ3t2YXJfbmFtZX0gPSAiIiJcbntlc2NhcGVkX2NvbnRleHR9XG4iIiJcblxuJykKICAgICAgICAKICAgICAgICAjIEdlbmVyYXRlIHF1ZXN0aW9ucyBkaWN0aW9uYXJ5CiAgICAgICAgZi53cml0ZSgnIyA9PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT1cbicpCiAgICAgICAgZi53cml0ZSgnIyBURVNUX1FVRVNUSU9OUyAtIFF1ZXN0aW9uLUFuc3dlciBwYWlycyBieSB0aXRsZVxuJykKICAgICAgICBmLndyaXRlKCcjID09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PVxuXG4nKQogICAgICAgIGYud3JpdGUoJ1RFU1RfUVVFU1RJT05TID0ge1xuJykKICAgICAgICAKICAgICAgICBmb3IgdGl0bGUsIHFhX3BhaXJzIGluIHF1ZXN0aW9ucy5pdGVtcygpOgogICAgICAgICAgICBpZiBub3QgcWFfcGFpcnM6CiAgICAgICAgICAgICAgICBjb250aW51ZQogICAgICAgICAgICAjIEVzY2FwZSBxdW90ZXMgaW4gdGl0bGUgKGJhY2tzbGFzaGVzIGZpcnN0LCB0aGVuIHF1b3RlcykKICAgICAgICAgICAgZXNjYXBlZF90aXRsZSA9IHRpdGxlLnJlcGxhY2UoJ1xcJywgJ1xcXFwnKS5yZXBsYWNlKCciJywgJ1xcIicpCiAgICAgICAgICAgIGYud3JpdGUoZicgICAgIntlc2NhcGVkX3RpdGxlfSI6IFtcbicpCiAgICAgICAgICAgIAogICAgICAgICAgICBmb3IgcXVlc3Rpb24sIGFuc3dlciBpbiBxYV9wYWlyczoKICAgICAgICAgICAgICAgICMgRXNjYXBlIGJhY2tzbGFzaGVzIGZpcnN0LCB0aGVuIHF1b3RlcyBhbmQgbmV3bGluZXMKICAgICAgICAgICAgICAgIGVzY2FwZWRfcSA9IHF1ZXN0aW9uLnJlcGxhY2UoJ1xcJywgJ1xcXFwnKS5yZXBsYWNlKCciJywgJ1xcIicpLnJlcGxhY2UoJ1xuJywgJyAnKQogICAgICAgICAgICAgICAgZXNjYXBlZF9hID0gYW5zd2VyLnJlcGxhY2UoJ1xcJywgJ1xcXFwnKS5yZXBsYWNlKCciJywgJ1xcIicpLnJlcGxhY2UoJ1xuJywgJyAnKQogICAgICAgICAgICAgICAgZi53cml0ZShmJyAgICAgICAgKCJ7ZXNjYXBlZF9xfSIsICJ7ZXNjYXBlZF9hfSIpLFxuJykKICAgICAgICAgICAgCiAgICAgICAgICAgIGYud3JpdGUoJyAgICBdLFxuJykKICAgICAgICAKICAgICAgICBmLndyaXRlKCd9XG5cbicpCiAgICAgICAgCiAgICAgICAgIyBIZWxwZXIgZnVuY3Rpb25zCiAgICAgICAgZi53cml0ZSgnIyA9PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT1cbicpCiAgICAgICAgZi53cml0ZSgnIyBIRUxQRVIgRlVOQ1RJT05TXG4nKQogICAgICAgIGYud3JpdGUoJyMgPT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09XG5cbicpCiAgICAgICAgCiAgICAgICAgZi53cml0ZSgnZGVmIGdldF9hbGxfY29udGV4dHMoKTpcbicpCiAgICAgICAgZi53cml0ZSgnICAgICIiIlJldHVybiBhbGwgY29udGV4dHMgYXMgYSBkaWN0aW9uYXJ5LiIiIlxuJykKICAgICAgICBmLndyaXRlKCcgICAgcmV0dXJuIHtcbicpCiAgICAgICAgZm9yIHRpdGxlIGluIGNvbnRleHRzLmtleXMoKToKICAgICAgICAgICAgdmFyX25hbWUgPSBjcmVhdGVfdmFyaWFibGVfbmFtZSh0aXRsZSkKICAgICAgICAgICAgZXNjYXBlZF90aXRsZSA9IHRpdGxlLnJlcGxhY2UoJ1xcJywgJ1xcXFwnKS5yZXBsYWNlKCciJywgJ1xcIicpCiAgICAgICAgICAgIGYud3JpdGUoZicgICAgICAgICJ7ZXNjYXBlZF90aXRsZX0iOiB7dmFyX25hbWV9LFxuJykKICAgICAgICBmLndyaXRlKCcgICAgfVxuXG4nKQogICAgICAgIAogICAgICAgIGYud3JpdGUoJ2RlZiBnZXRfdGVzdF9xdWVzdGlvbnMoKTpcbicpCiAgICAgICAgZi53cml0ZSgnICAgICIiIlJldHVybiB0ZXN0IHF1ZXN0aW9ucyB3aXRoIGV4cGVjdGVkIGFuc3dlcnMuIiIiXG4nKQogICAgICAgIGYud3JpdGUoJyAgICByZXR1cm4gVEVTVF9RVUVTVElPTlNcbicpCiAgICAKICAgIHByaW50KGYiT3V0cHV0IHNhdmVkIHRvOiB7b3V0cHV0X3BhdGh9IikKCgpkZWYgY3JlYXRlX3ZhcmlhYmxlX25hbWUodGl0bGU6IHN0cikgLT4gc3RyOgogICAgIiIiCiAgICBDcmVhdGUgYSB2YWxpZCBQeXRob24gdmFyaWFibGUgbmFtZSBmcm9tIHRpdGxlLgogICAgCiAgICBBcmdzOgogICAgICAgIHRpdGxlOiBPcmlnaW5hbCB0aXRsZSBzdHJpbmcKICAgICAgICAKICAgIFJldHVybnM6CiAgICAgICAgVmFsaWQgUHl0aG9uIHZhcmlhYmxlIG5hbWUKICAgICIiIgogICAgIyBSZXBsYWNlIG5vbi1hbHBoYW51bWVyaWMgY2hhcmFjdGVycyB3aXRoIHVuZGVyc2NvcmUKICAgIG5hbWUgPSByZS5zdWIocidbXmEtekEtWjAtOV0nLCAnXycsIHRpdGxlKQogICAgIyBSZW1vdmUgY29uc2VjdXRpdmUgdW5kZXJzY29yZXMKICAgIG5hbWUgPSByZS5zdWIocidfKycsICdfJywgbmFtZSkKICAgICMgUmVtb3ZlIGxlYWRpbmcvdHJhaWxpbmcgdW5kZXJzY29yZXMKICAgIG5hbWUgPSBuYW1lLnN0cmlwKCdfJykKICAgICMgRW5zdXJlIHN0YXJ0cyB3aXRoIGxldHRlcgogICAgaWYgbmFtZSBhbmQgbmFtZVswXS5pc2RpZ2l0KCk6CiAgICAgICAgbmFtZSA9ICdDT05URVhUXycgKyBuYW1lCiAgICBlbHNlOgogICAgICAgIG5hbWUgPSAnQ09OVEVYVF8nICsgbmFtZS51cHBlcigpCiAgICByZXR1cm4gbmFtZQoKCmRlZiBtYWluKCk6CiAgICAiIiJNYWluIGZ1bmN0aW9uIHRvIHByb2Nlc3MgdGhlIGRhdGFzZXQuIiIiCiAgICAjIENvbmZpZ3VyYXRpb24gLSBhdXRvLWRldGVjdCBpbnB1dCBmb3JtYXQKICAgIElOUFVUX0pTT04gPSAidHJhaW4tdjIuMC5qc29uIgogICAgSU5QVVRfQ1NWID0gInRyYWluLmNzdiIKICAgIE9VVFBVVF9GSUxFID0gInByb2Nlc3NlZF9zcXVhZC5weSIKICAgIAogICAgIyBHZXQgc2NyaXB0IGRpcmVjdG9yeSBmb3IgcmVsYXRpdmUgcGF0aHMKICAgIHNjcmlwdF9kaXIgPSBvcy5wYXRoLmRpcm5hbWUob3MucGF0aC5hYnNwYXRoKF9fZmlsZV9fKSkKICAgIG91dHB1dF9wYXRoID0gb3MucGF0aC5qb2luKHNjcmlwdF9kaXIsIE9VVFBVVF9GSUxFKQogICAgCiAgICBwcmludCgiPSIgKiA2MCkKICAgIHByaW50KCJTUXVBRCBEYXRhc2V0IFByb2Nlc3NvciIpCiAgICBwcmludCgiPSIgKiA2MCkKICAgIAogICAgIyBUcnkgdG8gZmluZCBpbnB1dCBmaWxlIChKU09OIG9yIENTVikKICAgIGpzb25fcGF0aCA9IG9zLnBhdGguam9pbihzY3JpcHRfZGlyLCBJTlBVVF9KU09OKQogICAgY3N2X3BhdGggPSBvcy5wYXRoLmpvaW4oc2NyaXB0X2RpciwgSU5QVVRfQ1NWKQogICAgCiAgICAjIExvYWQgZGF0YSAtIHByZWZlciBKU09OIChvZmZpY2lhbCBTUXVBRCBmb3JtYXQpCiAgICBwcmludCgiXG5bMS8zXSBMb2FkaW5nIGRhdGEuLi4iKQogICAgaWYgb3MucGF0aC5leGlzdHMoanNvbl9wYXRoKToKICAgICAgICBwcmludChmIkZvdW5kIEpTT04gZmlsZToge0lOUFVUX0pTT059IikKICAgICAgICBkYXRhID0gbG9hZF9qc29uX2RhdGEoanNvbl9wYXRoKQogICAgZWxpZiBvcy5wYXRoLmV4aXN0cyhjc3ZfcGF0aCk6CiAgICAgICAgcHJpbnQoZiJGb3VuZCBDU1YgZmlsZToge0lOUFVUX0NTVn0iKQogICAgICAgIGRhdGEgPSBsb2FkX2Nzdl9kYXRhKGNzdl9wYXRoKQogICAgZWxzZToKICAgICAgICBwcmludChmIuKdjCBFcnJvcjogTm8gaW5wdXQgZmlsZSBmb3VuZCEiKQogICAgICAgIHByaW50KGYiICAgRXhwZWN0ZWQ6IHtJTlBVVF9KU09OfSBvciB7SU5QVVRfQ1NWfSBpbiB7c2NyaXB0X2Rpcn0iKQogICAgICAgIHByaW50KGYiXG4gICBUbyBkb3dubG9hZCBTUXVBRCB2Mi4wOiIpCiAgICAgICAgcHJpbnQoZiIgICB3Z2V0IGh0dHBzOi8vcmFqcHVya2FyLmdpdGh1Yi5pby9TUXVBRC1leHBsb3Jlci9kYXRhc2V0L3RyYWluLXYyLjAuanNvbiIpCiAgICAgICAgcmV0dXJuCiAgICAKICAgICMgUHJvY2VzcyBkYXRhCiAgICBwcmludCgiXG5bMi8zXSBQcm9jZXNzaW5nIGRhdGFzZXQuLi4iKQogICAgY29udGV4dHMsIHF1ZXN0aW9ucyA9IHByb2Nlc3NfZGF0YXNldChkYXRhKQogICAgCiAgICAjIEdlbmVyYXRlIG91dHB1dAogICAgcHJpbnQoIlxuWzMvM10gR2VuZXJhdGluZyBvdXRwdXQgZmlsZS4uLiIpCiAgICBnZW5lcmF0ZV9vdXRwdXRfZmlsZShjb250ZXh0cywgcXVlc3Rpb25zLCBvdXRwdXRfcGF0aCkKICAgIAogICAgIyBTdW1tYXJ5CiAgICBwcmludCgiXG4iICsgIj0iICogNjApCiAgICBwcmludCgiU1VNTUFSWSIpCiAgICBwcmludCgiPSIgKiA2MCkKICAgIHByaW50KGYiICAtIFVuaXF1ZSB0aXRsZXM6IHtsZW4oY29udGV4dHMpfSIpCiAgICBwcmludChmIiAgLSBUb3RhbCBRQSBwYWlyczoge3N1bShsZW4ocWEpIGZvciBxYSBpbiBxdWVzdGlvbnMudmFsdWVzKCkpfSIpCiAgICBwcmludChmIiAgLSBPdXRwdXQgZmlsZToge291dHB1dF9wYXRofSIpCiAgICBwcmludCgiPSIgKiA2MCkKCgppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOgogICAgbWFpbigpCg=='
    os.makedirs('data', exist_ok=True)
    with open('data/process_squad_data.py', 'wb') as f:
        f.write(base64.b64decode(script_b64))
    
    print('⚙️ Processing SQuAD data...')
    !python data/process_squad_data.py
    
    if os.path.exists('data/processed_squad.py'):
        size_mb = os.path.getsize('data/processed_squad.py') / (1024 * 1024)
        print(f'✅ SQuAD dataset generated ({size_mb:.1f} MB)')
    else:
        print('❌ Failed to generate SQuAD dataset')


## 3. Run Baseline Comparison (Sample Essays)

This cell runs comparison on the Climate essay with 50 epochs.

In [ ]:
# Run baseline comparison on sample essays
!python projects/hybrid_titans/compare_baselines.py --essay climate --epochs 50

## 4. Run Baseline Comparison (SQuAD Dataset)

This cell runs comparison on the SQuAD dataset.

**Parameters:**
- `--squad`: Enable SQuAD mode
- `--titles N`: Number of titles to evaluate (default: 10)
- `--max-questions N`: Max questions per title (default: 5)
- `--epochs N`: Digestion epochs (default: 50)
- `--verbose`: Show detailed output

In [ ]:
# Run baseline comparison on SQuAD dataset
# Uncomment the line below to run

!python projects/hybrid_titans/compare_baselines.py --squad --titles 5 --max-questions 50 --epochs 200 --verbose --save-results


## 5. Run Multi-Document Comparison

This mode digests multiple SQuAD articles into a **shared** Titan memory, enabling cross-document retrieval.

**Parameters:**
- `--multi-doc`: Enable multi-document mode
- `--group-size N`: Number of titles to group together (default: 5)
- `--max-questions N`: Max questions per title (default: 5)
- `--epochs N`: Digestion epochs (default: 50)

In [ ]:
# Run multi-document cross-article comparison
# Articles are digested into a shared memory, retrieval spans all chunks

!python projects/hybrid_titans/compare_baselines.py --multi-doc --group-size 5 --max-questions 5 --epochs 200 --verbose --save-results


## 5.5 Setup HotpotQA Dataset (Optional)

This cell will automatically download and generate the HotpotQA dataset.
HotpotQA is a **multi-hop QA** dataset requiring reasoning across multiple documents.

**Skip this cell if you only want to use Sample Essays or SQuAD mode.**

In [ ]:
import os
import base64

# Check if HotpotQA data already exists
if os.path.exists('data/processed_hotpotqa.py'):
    size_mb = os.path.getsize('data/processed_hotpotqa.py') / (1024 * 1024)
    print(f'✅ HotpotQA dataset already exists ({size_mb:.1f} MB)')
else:
    print('📥 Downloading HotpotQA dataset...')
    !wget -q --show-progress http://curtis.ml.cmu.edu/datasets/hotpot/hotpot_train_v1.1.json -O data/hotpot_train_v1.1.json
    
    # Write the processing script
    script_b64 = 'IiIiClNjcmlwdCB0byBjbGVhbiBhbmQgcHJvY2VzcyBIb3Rwb3RRQSBkYXRhc2V0IChKU09OIGZvcm1hdCkuCgpUaGlzIHNjcmlwdDoKMS4gRXh0cmFjdHMgc3VwcG9ydGluZyBkb2N1bWVudHMgZnJvbSBtdWx0aS1kb2N1bWVudCBjb250ZXh0cwoyLiBDb21iaW5lcyBzdXBwb3J0aW5nIGRvY3VtZW50cyBpbnRvIGEgdW5pZmllZCBjb250ZXh0CjMuIENyZWF0ZXMgdW5pcXVlIHRpdGxlcyBiYXNlZCBvbiBzdXBwb3J0aW5nIGZhY3RzCjQuIEV4dHJhY3RzIHF1ZXN0aW9uLWFuc3dlciBwYWlycwoKT3V0cHV0IGZvcm1hdCBtYXRjaGVzIHNhbXBsZV9lc3NheXMucHkgYW5kIHByb2Nlc3NlZF9zcXVhZC5weSBzdHJ1Y3R1cmUuCgpLZXkgZGlmZmVyZW5jZXMgZnJvbSBTcXVhZCAyLjA6Ci0gTXVsdGktZG9jdW1lbnQgY29udGV4dHMgKG5lZWQgdG8gY29tYmluZSkKLSBTdXBwb3J0aW5nIGZhY3RzIGFubm90YXRpb24gKHVzZSB0byBmaWx0ZXIgZGlzdHJhY3RvcnMpCi0gWWVzL05vIGFuc3dlcnMgaW4gYWRkaXRpb24gdG8gdGV4dCBleHRyYWN0aW9uCi0gTXVsdGktaG9wIHJlYXNvbmluZyBxdWVzdGlvbnMKCkRvd25sb2FkIEhvdHBvdFFBIGRhdGFzZXQ6CiAgICB3Z2V0IGh0dHA6Ly9jdXJ0aXMubWwuY211LmVkdS9kYXRhc2V0cy9ob3Rwb3QvaG90cG90X3RyYWluX3YxLjEuanNvbgogICAgd2dldCBodHRwOi8vY3VydGlzLm1sLmNtdS5lZHUvZGF0YXNldHMvaG90cG90L2hvdHBvdF9kZXZfZGlzdHJhY3Rvcl92MS5qc29uCiIiIgoKaW1wb3J0IGpzb24KaW1wb3J0IHJlCmltcG9ydCBvcwppbXBvcnQgc3lzCmZyb20gY29sbGVjdGlvbnMgaW1wb3J0IGRlZmF1bHRkaWN0CgoKZGVmIGxvYWRfaG90cG90cWFfanNvbihmaWxlX3BhdGg6IHN0cikgLT4gbGlzdDoKICAgICIiIgogICAgTG9hZCBIb3Rwb3RRQSBKU09OIGRhdGFzZXQuCiAgICAKICAgIEFyZ3M6CiAgICAgICAgZmlsZV9wYXRoOiBQYXRoIHRvIHRoZSBKU09OIGZpbGUgKGhvdHBvdF90cmFpbl92MS4xLmpzb24gb3IgaG90cG90X2Rldl9kaXN0cmFjdG9yX3YxLmpzb24pCiAgICAgICAgCiAgICBSZXR1cm5zOgogICAgICAgIExpc3Qgb2YgZGF0YSBpdGVtcywgZWFjaCBjb250YWluaW5nOgogICAgICAgIC0gX2lkOiB1bmlxdWUgaWRlbnRpZmllcgogICAgICAgIC0gcXVlc3Rpb246IHRoZSBxdWVzdGlvbiB0ZXh0CiAgICAgICAgLSBhbnN3ZXI6IGFuc3dlciAodGV4dCBvciB5ZXMvbm8pCiAgICAgICAgLSBzdXBwb3J0aW5nX2ZhY3RzOiBbW3RpdGxlLCBzZW50X2lkXSwgLi4uXQogICAgICAgIC0gY29udGV4dDogW1t0aXRsZSwgW3NlbnQxLCBzZW50MiwgLi4uXV0sIC4uLl0KICAgICAgICAtIHR5cGU6IHF1ZXN0aW9uIHR5cGUgKGNvbXBhcmlzb24sIGJyaWRnZSwgZXRjLikKICAgICAgICAtIGxldmVsOiBkaWZmaWN1bHR5IChlYXN5LCBtZWRpdW0sIGhhcmQpCiAgICAiIiIKICAgIHdpdGggb3BlbihmaWxlX3BhdGgsICdyJywgZW5jb2Rpbmc9J3V0Zi04JykgYXMgZjoKICAgICAgICBkYXRhID0ganNvbi5sb2FkKGYpCiAgICAKICAgIHByaW50KGYiTG9hZGVkIHtsZW4oZGF0YSl9IGl0ZW1zIGZyb20gSG90cG90UUEgZGF0YXNldCIpCiAgICByZXR1cm4gZGF0YQoKCmRlZiBleHRyYWN0X3N1cHBvcnRpbmdfY29udGV4dChpdGVtOiBkaWN0LCBpbmNsdWRlX2FsbDogYm9vbCA9IEZhbHNlKSAtPiBzdHI6CiAgICAiIiIKICAgIEV4dHJhY3QgYW5kIGNvbWJpbmUgY29udGV4dCBmcm9tIHN1cHBvcnRpbmcgZG9jdW1lbnRzLgogICAgCiAgICBBcmdzOgogICAgICAgIGl0ZW06IEhvdHBvdFFBIGRhdGEgaXRlbQogICAgICAgIGluY2x1ZGVfYWxsOiBJZiBUcnVlLCBpbmNsdWRlIGFsbCBkb2N1bWVudHMgKGluY2x1ZGluZyBkaXN0cmFjdG9ycykKICAgICAgICAgICAgICAgICAgICBJZiBGYWxzZSwgb25seSBpbmNsdWRlIHN1cHBvcnRpbmcgZG9jdW1lbnRzCiAgICAgICAgCiAgICBSZXR1cm5zOgogICAgICAgIENvbWJpbmVkIGNvbnRleHQgc3RyaW5nIHdpdGggZG9jdW1lbnQgdGl0bGVzIGFzIGhlYWRlcnMKICAgICIiIgogICAgaWYgaW5jbHVkZV9hbGw6CiAgICAgICAgIyBJbmNsdWRlIGFsbCBkb2N1bWVudHMKICAgICAgICBjb250ZXh0X3BhcnRzID0gW10KICAgICAgICBmb3IgZG9jX3RpdGxlLCBzZW50ZW5jZXMgaW4gaXRlbVsnY29udGV4dCddOgogICAgICAgICAgICBwYXJhZ3JhcGggPSAiICIuam9pbihzZW50ZW5jZXMpCiAgICAgICAgICAgIGNvbnRleHRfcGFydHMuYXBwZW5kKGYiIyB7ZG9jX3RpdGxlfVxue3BhcmFncmFwaH0iKQogICAgZWxzZToKICAgICAgICAjIE9ubHkgaW5jbHVkZSBzdXBwb3J0aW5nIGRvY3VtZW50cwogICAgICAgIHN1cHBvcnRpbmdfdGl0bGVzID0gc2V0KFt0aXRsZSBmb3IgdGl0bGUsIF8gaW4gaXRlbVsnc3VwcG9ydGluZ19mYWN0cyddXSkKICAgICAgICBjb250ZXh0X3BhcnRzID0gW10KICAgICAgICAKICAgICAgICBmb3IgZG9jX3RpdGxlLCBzZW50ZW5jZXMgaW4gaXRlbVsnY29udGV4dCddOgogICAgICAgICAgICBpZiBkb2NfdGl0bGUgaW4gc3VwcG9ydGluZ190aXRsZXM6CiAgICAgICAgICAgICAgICBwYXJhZ3JhcGggPSAiICIuam9pbihzZW50ZW5jZXMpCiAgICAgICAgICAgICAgICBjb250ZXh0X3BhcnRzLmFwcGVuZChmIiMge2RvY190aXRsZX1cbntwYXJhZ3JhcGh9IikKICAgIAogICAgcmV0dXJuICJcblxuIi5qb2luKGNvbnRleHRfcGFydHMpCgoKZGVmIGNyZWF0ZV91bmlxdWVfdGl0bGUoaXRlbTogZGljdCkgLT4gc3RyOgogICAgIiIiCiAgICBDcmVhdGUgYSB1bmlxdWUgdGl0bGUgZm9yIGVhY2ggcXVlc3Rpb24gYmFzZWQgb24gc3VwcG9ydGluZyBmYWN0cy4KICAgIAogICAgQXJnczoKICAgICAgICBpdGVtOiBIb3Rwb3RRQSBkYXRhIGl0ZW0KICAgICAgICAKICAgIFJldHVybnM6CiAgICAgICAgVW5pcXVlIHRpdGxlIHN0cmluZyBjb21iaW5pbmcgc3VwcG9ydGluZyBkb2N1bWVudCB0aXRsZXMKICAgICIiIgogICAgIyBFeHRyYWN0IHN1cHBvcnRpbmcgZG9jdW1lbnQgdGl0bGVzCiAgICBzdXBwb3J0aW5nX3RpdGxlcyA9IFtdCiAgICBzZWVuID0gc2V0KCkKICAgIGZvciB0aXRsZSwgXyBpbiBpdGVtWydzdXBwb3J0aW5nX2ZhY3RzJ106CiAgICAgICAgaWYgdGl0bGUgbm90IGluIHNlZW46CiAgICAgICAgICAgIHN1cHBvcnRpbmdfdGl0bGVzLmFwcGVuZCh0aXRsZSkKICAgICAgICAgICAgc2Vlbi5hZGQodGl0bGUpCiAgICAKICAgICMgQ29tYmluZSB0aXRsZXMgKG1heCAzIHRvIGtlZXAgdmFyaWFibGUgbmFtZXMgcmVhc29uYWJsZSkKICAgIGlmIGxlbihzdXBwb3J0aW5nX3RpdGxlcykgPD0gMzoKICAgICAgICByZXR1cm4gIiBhbmQgIi5qb2luKHN1cHBvcnRpbmdfdGl0bGVzKQogICAgZWxzZToKICAgICAgICByZXR1cm4gIiBhbmQgIi5qb2luKHN1cHBvcnRpbmdfdGl0bGVzWzozXSkgKyAiIGV0IGFsIgoKCmRlZiBwcm9jZXNzX2hvdHBvdHFhX2RhdGFzZXQoZGF0YTogbGlzdCwgCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgaW5jbHVkZV9hbGxfZG9jczogYm9vbCA9IEZhbHNlLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgIG1heF9pdGVtczogaW50ID0gTm9uZSkgLT4gdHVwbGU6CiAgICAiIiIKICAgIFByb2Nlc3MgSG90cG90UUEgZGF0YXNldCBpbnRvIGNvbnRleHRzIGFuZCBxdWVzdGlvbnMuCiAgICAKICAgIEFyZ3M6CiAgICAgICAgZGF0YTogTGlzdCBvZiBIb3Rwb3RRQSBpdGVtcwogICAgICAgIGluY2x1ZGVfYWxsX2RvY3M6IElmIFRydWUsIGluY2x1ZGUgZGlzdHJhY3RvciBkb2N1bWVudHMKICAgICAgICBtYXhfaXRlbXM6IE1heGltdW0gbnVtYmVyIG9mIGl0ZW1zIHRvIHByb2Nlc3MgKGZvciB0ZXN0aW5nKQogICAgICAgIAogICAgUmV0dXJuczoKICAgICAgICBUdXBsZSBvZiAoY29udGV4dHNfZGljdCwgcXVlc3Rpb25zX2RpY3QsIG1ldGFkYXRhX2RpY3QpCiAgICAgICAgLSBjb250ZXh0c19kaWN0OiB7dGl0bGU6IGNvbWJpbmVkX2NvbnRleHR9CiAgICAgICAgLSBxdWVzdGlvbnNfZGljdDoge3RpdGxlOiBbKHF1ZXN0aW9uLCBhbnN3ZXIpLCAuLi5dfQogICAgICAgIC0gbWV0YWRhdGFfZGljdDoge3RpdGxlOiB7J3R5cGUnOiBbLi4uXSwgJ2xldmVsJzogWy4uLl0sICdpZHMnOiBbLi4uXX19CiAgICAiIiIKICAgICMgVXNlIGRlZmF1bHRkaWN0IHRvIGNvbGxlY3QgaXRlbXMgd2l0aCBzYW1lIHN1cHBvcnRpbmcgZG9jdW1lbnRzCiAgICB0aXRsZV90b19pdGVtcyA9IGRlZmF1bHRkaWN0KGxpc3QpCiAgICAKICAgICMgTGltaXQgaXRlbXMgaWYgc3BlY2lmaWVkCiAgICBpZiBtYXhfaXRlbXM6CiAgICAgICAgZGF0YSA9IGRhdGFbOm1heF9pdGVtc10KICAgIAogICAgIyBHcm91cCBpdGVtcyBieSB1bmlxdWUgdGl0bGUKICAgIGZvciBpdGVtIGluIGRhdGE6CiAgICAgICAgdGl0bGUgPSBjcmVhdGVfdW5pcXVlX3RpdGxlKGl0ZW0pCiAgICAgICAgdGl0bGVfdG9faXRlbXNbdGl0bGVdLmFwcGVuZChpdGVtKQogICAgCiAgICAjIFByb2Nlc3MgZWFjaCBncm91cAogICAgY29udGV4dHNfZGljdCA9IHt9CiAgICBxdWVzdGlvbnNfZGljdCA9IHt9CiAgICBtZXRhZGF0YV9kaWN0ID0ge30KICAgIAogICAgZm9yIHRpdGxlLCBpdGVtcyBpbiB0aXRsZV90b19pdGVtcy5pdGVtcygpOgogICAgICAgICMgVXNlIHRoZSBmaXJzdCBpdGVtJ3MgY29udGV4dCAodGhleSBzaG91bGQgYmUgc2ltaWxhciBpZiBzdXBwb3J0aW5nIGRvY3MgYXJlIHRoZSBzYW1lKQogICAgICAgIGNvbnRleHQgPSBleHRyYWN0X3N1cHBvcnRpbmdfY29udGV4dChpdGVtc1swXSwgaW5jbHVkZV9hbGw9aW5jbHVkZV9hbGxfZG9jcykKICAgICAgICBjb250ZXh0c19kaWN0W3RpdGxlXSA9IGNvbnRleHQKICAgICAgICAKICAgICAgICAjIENvbGxlY3QgYWxsIFEmQSBwYWlycwogICAgICAgIHFhX3BhaXJzID0gW10KICAgICAgICB0eXBlcyA9IFtdCiAgICAgICAgbGV2ZWxzID0gW10KICAgICAgICBpZHMgPSBbXQogICAgICAgIAogICAgICAgIGZvciBpdGVtIGluIGl0ZW1zOgogICAgICAgICAgICBxdWVzdGlvbiA9IGl0ZW1bJ3F1ZXN0aW9uJ10KICAgICAgICAgICAgYW5zd2VyID0gaXRlbVsnYW5zd2VyJ10KICAgICAgICAgICAgcWFfcGFpcnMuYXBwZW5kKChxdWVzdGlvbiwgYW5zd2VyKSkKICAgICAgICAgICAgCiAgICAgICAgICAgICMgQ29sbGVjdCBtZXRhZGF0YQogICAgICAgICAgICB0eXBlcy5hcHBlbmQoaXRlbS5nZXQoJ3R5cGUnLCAndW5rbm93bicpKQogICAgICAgICAgICBsZXZlbHMuYXBwZW5kKGl0ZW0uZ2V0KCdsZXZlbCcsICd1bmtub3duJykpCiAgICAgICAgICAgIGlkcy5hcHBlbmQoaXRlbVsnX2lkJ10pCiAgICAgICAgCiAgICAgICAgcXVlc3Rpb25zX2RpY3RbdGl0bGVdID0gcWFfcGFpcnMKICAgICAgICBtZXRhZGF0YV9kaWN0W3RpdGxlXSA9IHsKICAgICAgICAgICAgJ3R5cGUnOiB0eXBlcywKICAgICAgICAgICAgJ2xldmVsJzogbGV2ZWxzLAogICAgICAgICAgICAnaWRzJzogaWRzCiAgICAgICAgfQogICAgCiAgICBwcmludChmIlByb2Nlc3NlZCB7bGVuKGNvbnRleHRzX2RpY3QpfSB1bmlxdWUgY29udGV4dCBncm91cHMiKQogICAgcHJpbnQoZiJUb3RhbCBRQSBwYWlyczoge3N1bShsZW4ocWEpIGZvciBxYSBpbiBxdWVzdGlvbnNfZGljdC52YWx1ZXMoKSl9IikKICAgIAogICAgcmV0dXJuIGNvbnRleHRzX2RpY3QsIHF1ZXN0aW9uc19kaWN0LCBtZXRhZGF0YV9kaWN0CgoKZGVmIGNyZWF0ZV92YXJpYWJsZV9uYW1lKHRpdGxlOiBzdHIpIC0+IHN0cjoKICAgICIiIgogICAgQ3JlYXRlIGEgdmFsaWQgUHl0aG9uIHZhcmlhYmxlIG5hbWUgZnJvbSB0aXRsZS4KICAgIAogICAgQXJnczoKICAgICAgICB0aXRsZTogT3JpZ2luYWwgdGl0bGUgc3RyaW5nCiAgICAgICAgCiAgICBSZXR1cm5zOgogICAgICAgIFZhbGlkIFB5dGhvbiB2YXJpYWJsZSBuYW1lCiAgICAiIiIKICAgICMgUmVwbGFjZSBub24tYWxwaGFudW1lcmljIGNoYXJhY3RlcnMgd2l0aCB1bmRlcnNjb3JlCiAgICBuYW1lID0gcmUuc3ViKHInW15hLXpBLVowLTldJywgJ18nLCB0aXRsZSkKICAgICMgUmVtb3ZlIGNvbnNlY3V0aXZlIHVuZGVyc2NvcmVzCiAgICBuYW1lID0gcmUuc3ViKHInXysnLCAnXycsIG5hbWUpCiAgICAjIFJlbW92ZSBsZWFkaW5nL3RyYWlsaW5nIHVuZGVyc2NvcmVzCiAgICBuYW1lID0gbmFtZS5zdHJpcCgnXycpCiAgICAjIEVuc3VyZSBzdGFydHMgd2l0aCBsZXR0ZXIgYW5kIGFkZCBwcmVmaXgKICAgIGlmIG5hbWUgYW5kIG5hbWVbMF0uaXNkaWdpdCgpOgogICAgICAgIG5hbWUgPSAnQ09OVEVYVF8nICsgbmFtZQogICAgZWxzZToKICAgICAgICBuYW1lID0gJ0NPTlRFWFRfJyArIG5hbWUudXBwZXIoKQogICAgcmV0dXJuIG5hbWUKCgpkZWYgZ2VuZXJhdGVfb3V0cHV0X2ZpbGUoY29udGV4dHM6IGRpY3QsIAogICAgICAgICAgICAgICAgICAgICAgICAgcXVlc3Rpb25zOiBkaWN0LCAKICAgICAgICAgICAgICAgICAgICAgICAgIG1ldGFkYXRhOiBkaWN0LAogICAgICAgICAgICAgICAgICAgICAgICAgb3V0cHV0X3BhdGg6IHN0ciwKICAgICAgICAgICAgICAgICAgICAgICAgIGluY2x1ZGVfbWV0YWRhdGE6IGJvb2wgPSBUcnVlKToKICAgICIiIgogICAgR2VuZXJhdGUgb3V0cHV0IFB5dGhvbiBmaWxlIGluIHNhbXBsZV9lc3NheXMucHkgZm9ybWF0LgogICAgCiAgICBBcmdzOgogICAgICAgIGNvbnRleHRzOiBEaWN0aW9uYXJ5IG9mIHt0aXRsZTogY29tYmluZWRfY29udGV4dH0KICAgICAgICBxdWVzdGlvbnM6IERpY3Rpb25hcnkgb2Yge3RpdGxlOiBbKHF1ZXN0aW9uLCBhbnN3ZXIpLCAuLi5dfQogICAgICAgIG1ldGFkYXRhOiBEaWN0aW9uYXJ5IG9mIHt0aXRsZTogeyd0eXBlJzogWy4uLl0sICdsZXZlbCc6IFsuLi5dLCAnaWRzJzogWy4uLl19fQogICAgICAgIG91dHB1dF9wYXRoOiBQYXRoIGZvciBvdXRwdXQgZmlsZQogICAgICAgIGluY2x1ZGVfbWV0YWRhdGE6IFdoZXRoZXIgdG8gaW5jbHVkZSBtZXRhZGF0YSBjb21tZW50cwogICAgIiIiCiAgICB3aXRoIG9wZW4ob3V0cHV0X3BhdGgsICd3JywgZW5jb2Rpbmc9J3V0Zi04JykgYXMgZjoKICAgICAgICAjIEhlYWRlcgogICAgICAgIGYud3JpdGUoJyIiIlxuJykKICAgICAgICBmLndyaXRlKCdQcm9jZXNzZWQgSG90cG90UUEgRGF0YXNldCBmb3IgVGl0YW5SQUcgRXhwZXJpbWVudHNcbicpCiAgICAgICAgZi53cml0ZSgnXG4nKQogICAgICAgIGYud3JpdGUoJ1RoaXMgZmlsZSBjb250YWlucyBwcm9jZXNzZWQgY29udGV4dHMgYW5kIHF1ZXN0aW9uLWFuc3dlciBwYWlyc1xuJykKICAgICAgICBmLndyaXRlKCdleHRyYWN0ZWQgZnJvbSB0aGUgSG90cG90UUEgZGF0YXNldC5cbicpCiAgICAgICAgZi53cml0ZSgnXG4nKQogICAgICAgIGYud3JpdGUoJ05vdGU6IEhvdHBvdFFBIGlzIGEgbXVsdGktaG9wIFFBIGRhdGFzZXQgcmVxdWlyaW5nIHJlYXNvbmluZ1xuJykKICAgICAgICBmLndyaXRlKCdhY3Jvc3MgbXVsdGlwbGUgZG9jdW1lbnRzLiBDb250ZXh0cyBjb21iaW5lIHN1cHBvcnRpbmcgZG9jdW1lbnRzLlxuJykKICAgICAgICBmLndyaXRlKCciIiJcblxuJykKICAgICAgICAKICAgICAgICAjIEdlbmVyYXRlIGNvbnRleHQgdmFyaWFibGVzCiAgICAgICAgZi53cml0ZSgnIyA9PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT1cbicpCiAgICAgICAgZi53cml0ZSgnIyBDT05URVhUUyAtIENvbWJpbmVkIHN1cHBvcnRpbmcgZG9jdW1lbnRzXG4nKQogICAgICAgIGYud3JpdGUoJyMgPT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09XG5cbicpCiAgICAgICAgCiAgICAgICAgZm9yIHRpdGxlLCBjb250ZXh0IGluIGNvbnRleHRzLml0ZW1zKCk6CiAgICAgICAgICAgIHZhcl9uYW1lID0gY3JlYXRlX3ZhcmlhYmxlX25hbWUodGl0bGUpCiAgICAgICAgICAgIAogICAgICAgICAgICAjIEFkZCBtZXRhZGF0YSBjb21tZW50IGlmIHJlcXVlc3RlZAogICAgICAgICAgICBpZiBpbmNsdWRlX21ldGFkYXRhIGFuZCB0aXRsZSBpbiBtZXRhZGF0YToKICAgICAgICAgICAgICAgIG1ldGEgPSBtZXRhZGF0YVt0aXRsZV0KICAgICAgICAgICAgICAgIGYud3JpdGUoZicjIFF1ZXN0aW9uIHR5cGVzOiB7IiwgIi5qb2luKHNldChtZXRhWyJ0eXBlIl0pKX1cbicpCiAgICAgICAgICAgICAgICBmLndyaXRlKGYnIyBEaWZmaWN1bHR5IGxldmVsczogeyIsICIuam9pbihzZXQobWV0YVsibGV2ZWwiXSkpfVxuJykKICAgICAgICAgICAgICAgIGYud3JpdGUoZicjIE51bWJlciBvZiBxdWVzdGlvbnM6IHtsZW4ocXVlc3Rpb25zW3RpdGxlXSl9XG4nKQogICAgICAgICAgICAKICAgICAgICAgICAgIyBFc2NhcGUgYmFja3NsYXNoZXMgZmlyc3QsIHRoZW4gdHJpcGxlIHF1b3RlcyBpbiBjb250ZW50CiAgICAgICAgICAgIGVzY2FwZWRfY29udGV4dCA9IGNvbnRleHQucmVwbGFjZSgnXFwnLCAnXFxcXCcpLnJlcGxhY2UoJyIiIicsICdcXCJcXCJcXCInKQogICAgICAgICAgICBmLndyaXRlKGYne3Zhcl9uYW1lfSA9ICIiIlxue2VzY2FwZWRfY29udGV4dH1cbiIiIlxuXG4nKQogICAgICAgIAogICAgICAgICMgR2VuZXJhdGUgcXVlc3Rpb25zIGRpY3Rpb25hcnkKICAgICAgICBmLndyaXRlKCcjID09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PVxuJykKICAgICAgICBmLndyaXRlKCcjIFRFU1RfUVVFU1RJT05TIC0gUXVlc3Rpb24tQW5zd2VyIHBhaXJzXG4nKQogICAgICAgIGYud3JpdGUoJyMgPT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09XG5cbicpCiAgICAgICAgZi53cml0ZSgnVEVTVF9RVUVTVElPTlMgPSB7XG4nKQogICAgICAgIAogICAgICAgIGZvciB0aXRsZSwgcWFfcGFpcnMgaW4gcXVlc3Rpb25zLml0ZW1zKCk6CiAgICAgICAgICAgIGlmIG5vdCBxYV9wYWlyczoKICAgICAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgICAgIAogICAgICAgICAgICAjIEVzY2FwZSBxdW90ZXMgaW4gdGl0bGUgKGJhY2tzbGFzaGVzIGZpcnN0LCB0aGVuIHF1b3RlcykKICAgICAgICAgICAgZXNjYXBlZF90aXRsZSA9IHRpdGxlLnJlcGxhY2UoJ1xcJywgJ1xcXFwnKS5yZXBsYWNlKCciJywgJ1xcIicpCiAgICAgICAgICAgIGYud3JpdGUoZicgICAgIntlc2NhcGVkX3RpdGxlfSI6IFtcbicpCiAgICAgICAgICAgIAogICAgICAgICAgICBmb3IgcXVlc3Rpb24sIGFuc3dlciBpbiBxYV9wYWlyczoKICAgICAgICAgICAgICAgICMgRXNjYXBlIGJhY2tzbGFzaGVzIGZpcnN0LCB0aGVuIHF1b3RlcyBhbmQgbmV3bGluZXMKICAgICAgICAgICAgICAgIGVzY2FwZWRfcSA9IHF1ZXN0aW9uLnJlcGxhY2UoJ1xcJywgJ1xcXFwnKS5yZXBsYWNlKCciJywgJ1xcIicpLnJlcGxhY2UoJ1xuJywgJyAnKQogICAgICAgICAgICAgICAgZXNjYXBlZF9hID0gYW5zd2VyLnJlcGxhY2UoJ1xcJywgJ1xcXFwnKS5yZXBsYWNlKCciJywgJ1xcIicpLnJlcGxhY2UoJ1xuJywgJyAnKQogICAgICAgICAgICAgICAgZi53cml0ZShmJyAgICAgICAgKCJ7ZXNjYXBlZF9xfSIsICJ7ZXNjYXBlZF9hfSIpLFxuJykKICAgICAgICAgICAgCiAgICAgICAgICAgIGYud3JpdGUoJyAgICBdLFxuJykKICAgICAgICAKICAgICAgICBmLndyaXRlKCd9XG5cbicpCiAgICAgICAgCiAgICAgICAgIyBPcHRpb25hbDogSW5jbHVkZSBtZXRhZGF0YSBkaWN0aW9uYXJ5CiAgICAgICAgaWYgaW5jbHVkZV9tZXRhZGF0YToKICAgICAgICAgICAgZi53cml0ZSgnIyA9PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT1cbicpCiAgICAgICAgICAgIGYud3JpdGUoJyMgTUVUQURBVEEgLSBRdWVzdGlvbiB0eXBlcyBhbmQgZGlmZmljdWx0eSBsZXZlbHNcbicpCiAgICAgICAgICAgIGYud3JpdGUoJyMgPT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09XG5cbicpCiAgICAgICAgICAgIGYud3JpdGUoJ1FVRVNUSU9OX01FVEFEQVRBID0ge1xuJykKICAgICAgICAgICAgCiAgICAgICAgICAgIGZvciB0aXRsZSwgbWV0YSBpbiBtZXRhZGF0YS5pdGVtcygpOgogICAgICAgICAgICAgICAgZXNjYXBlZF90aXRsZSA9IHRpdGxlLnJlcGxhY2UoJ1xcJywgJ1xcXFwnKS5yZXBsYWNlKCciJywgJ1xcIicpCiAgICAgICAgICAgICAgICBmLndyaXRlKGYnICAgICJ7ZXNjYXBlZF90aXRsZX0iOiB7e1xuJykKICAgICAgICAgICAgICAgIGYud3JpdGUoZicgICAgICAgICJ0eXBlcyI6IHtsaXN0KHNldChtZXRhWyJ0eXBlIl0pKX0sXG4nKQogICAgICAgICAgICAgICAgZi53cml0ZShmJyAgICAgICAgImxldmVscyI6IHtsaXN0KHNldChtZXRhWyJsZXZlbCJdKSl9LFxuJykKICAgICAgICAgICAgICAgIGYud3JpdGUoZicgICAgICAgICJpZHMiOiB7bWV0YVsiaWRzIl19LFxuJykKICAgICAgICAgICAgICAgIGYud3JpdGUoJyAgICB9LFxuJykKICAgICAgICAgICAgCiAgICAgICAgICAgIGYud3JpdGUoJ31cblxuJykKICAgICAgICAKICAgICAgICAjIEhlbHBlciBmdW5jdGlvbnMKICAgICAgICBmLndyaXRlKCcjID09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PVxuJykKICAgICAgICBmLndyaXRlKCcjIEhFTFBFUiBGVU5DVElPTlNcbicpCiAgICAgICAgZi53cml0ZSgnIyA9PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT1cblxuJykKICAgICAgICAKICAgICAgICBmLndyaXRlKCdkZWYgZ2V0X2FsbF9jb250ZXh0cygpOlxuJykKICAgICAgICBmLndyaXRlKCcgICAgIiIiUmV0dXJuIGFsbCBjb250ZXh0cyBhcyBhIGRpY3Rpb25hcnkuIiIiXG4nKQogICAgICAgIGYud3JpdGUoJyAgICByZXR1cm4ge1xuJykKICAgICAgICBmb3IgdGl0bGUgaW4gY29udGV4dHMua2V5cygpOgogICAgICAgICAgICB2YXJfbmFtZSA9IGNyZWF0ZV92YXJpYWJsZV9uYW1lKHRpdGxlKQogICAgICAgICAgICBlc2NhcGVkX3RpdGxlID0gdGl0bGUucmVwbGFjZSgnXFwnLCAnXFxcXCcpLnJlcGxhY2UoJyInLCAnXFwiJykKICAgICAgICAgICAgZi53cml0ZShmJyAgICAgICAgIntlc2NhcGVkX3RpdGxlfSI6IHt2YXJfbmFtZX0sXG4nKQogICAgICAgIGYud3JpdGUoJyAgICB9XG5cbicpCiAgICAgICAgCiAgICAgICAgZi53cml0ZSgnZGVmIGdldF90ZXN0X3F1ZXN0aW9ucygpOlxuJykKICAgICAgICBmLndyaXRlKCcgICAgIiIiUmV0dXJuIHRlc3QgcXVlc3Rpb25zIHdpdGggZXhwZWN0ZWQgYW5zd2Vycy4iIiJcbicpCiAgICAgICAgZi53cml0ZSgnICAgIHJldHVybiBURVNUX1FVRVNUSU9OU1xuXG4nKQogICAgICAgIAogICAgICAgIGlmIGluY2x1ZGVfbWV0YWRhdGE6CiAgICAgICAgICAgIGYud3JpdGUoJ2RlZiBnZXRfcXVlc3Rpb25fbWV0YWRhdGEoKTpcbicpCiAgICAgICAgICAgIGYud3JpdGUoJyAgICAiIiJSZXR1cm4gbWV0YWRhdGEgZm9yIHF1ZXN0aW9ucyAodHlwZXMsIGxldmVscywgSURzKS4iIiJcbicpCiAgICAgICAgICAgIGYud3JpdGUoJyAgICByZXR1cm4gUVVFU1RJT05fTUVUQURBVEFcbicpCiAgICAKICAgIHByaW50KGYiT3V0cHV0IHNhdmVkIHRvOiB7b3V0cHV0X3BhdGh9IikKCgpkZWYgbWFpbigpOgogICAgIiIiTWFpbiBmdW5jdGlvbiB0byBwcm9jZXNzIEhvdHBvdFFBIGRhdGFzZXQuIiIiCiAgICAjIENvbmZpZ3VyYXRpb24KICAgIElOUFVUX0ZJTEUgPSAiaG90cG90X3RyYWluX3YxLjEuanNvbiIgICMgb3IgaG90cG90X2Rldl9kaXN0cmFjdG9yX3YxLmpzb24KICAgIE9VVFBVVF9GSUxFID0gInByb2Nlc3NlZF9ob3Rwb3RxYS5weSIKICAgIAogICAgIyBQcm9jZXNzaW5nIG9wdGlvbnMKICAgIElOQ0xVREVfQUxMX0RPQ1MgPSBGYWxzZSAgIyBGYWxzZSA9IG9ubHkgc3VwcG9ydGluZyBkb2NzLCBUcnVlID0gaW5jbHVkZSBkaXN0cmFjdG9ycwogICAgSU5DTFVERV9NRVRBREFUQSA9IFRydWUgICAjIEluY2x1ZGUgbWV0YWRhdGEgY29tbWVudHMgYW5kIGRpY3Rpb25hcnkKICAgIE1BWF9JVEVNUyA9IE5vbmUgICAgICAgICAgIyBTZXQgdG8gZS5nLiwgMTAwMCBmb3IgdGVzdGluZywgTm9uZSBmb3IgYWxsCiAgICAKICAgICMgR2V0IHNjcmlwdCBkaXJlY3RvcnkgZm9yIHJlbGF0aXZlIHBhdGhzCiAgICBzY3JpcHRfZGlyID0gb3MucGF0aC5kaXJuYW1lKG9zLnBhdGguYWJzcGF0aChfX2ZpbGVfXykpCiAgICBpbnB1dF9wYXRoID0gb3MucGF0aC5qb2luKHNjcmlwdF9kaXIsIElOUFVUX0ZJTEUpCiAgICBvdXRwdXRfcGF0aCA9IG9zLnBhdGguam9pbihzY3JpcHRfZGlyLCBPVVRQVVRfRklMRSkKICAgIAogICAgcHJpbnQoIj0iICogNjApCiAgICBwcmludCgiSG90cG90UUEgRGF0YXNldCBQcm9jZXNzb3IiKQogICAgcHJpbnQoIj0iICogNjApCiAgICBwcmludChmIk1vZGU6IHsnQWxsIGRvY3VtZW50cyAoaW5jbC4gZGlzdHJhY3RvcnMpJyBpZiBJTkNMVURFX0FMTF9ET0NTIGVsc2UgJ1N1cHBvcnRpbmcgZG9jdW1lbnRzIG9ubHknfSIpCiAgICBwcmludChmIk1ldGFkYXRhOiB7J0luY2x1ZGVkJyBpZiBJTkNMVURFX01FVEFEQVRBIGVsc2UgJ0V4Y2x1ZGVkJ30iKQogICAgaWYgTUFYX0lURU1TOgogICAgICAgIHByaW50KGYiUHJvY2Vzc2luZzogRmlyc3Qge01BWF9JVEVNU30gaXRlbXMiKQogICAgcHJpbnQoIj0iICogNjApCiAgICAKICAgICMgQ2hlY2sgaWYgaW5wdXQgZmlsZSBleGlzdHMKICAgIGlmIG5vdCBvcy5wYXRoLmV4aXN0cyhpbnB1dF9wYXRoKToKICAgICAgICBwcmludChmIlxu4p2MIEVycm9yOiBJbnB1dCBmaWxlIG5vdCBmb3VuZCEiKQogICAgICAgIHByaW50KGYiICAgRXhwZWN0ZWQ6IHtpbnB1dF9wYXRofSIpCiAgICAgICAgcHJpbnQoZiJcbiAgIFRvIGRvd25sb2FkIEhvdHBvdFFBIGRhdGFzZXQ6IikKICAgICAgICBwcmludChmIiAgIHdnZXQgaHR0cDovL2N1cnRpcy5tbC5jbXUuZWR1L2RhdGFzZXRzL2hvdHBvdC9ob3Rwb3RfdHJhaW5fdjEuMS5qc29uIikKICAgICAgICBwcmludChmIiAgIHdnZXQgaHR0cDovL2N1cnRpcy5tbC5jbXUuZWR1L2RhdGFzZXRzL2hvdHBvdC9ob3Rwb3RfZGV2X2Rpc3RyYWN0b3JfdjEuanNvbiIpCiAgICAgICAgcmV0dXJuCiAgICAKICAgICMgTG9hZCBkYXRhCiAgICBwcmludCgiXG5bMS8zXSBMb2FkaW5nIGRhdGEuLi4iKQogICAgZGF0YSA9IGxvYWRfaG90cG90cWFfanNvbihpbnB1dF9wYXRoKQogICAgCiAgICAjIFByb2Nlc3MgZGF0YQogICAgcHJpbnQoIlxuWzIvM10gUHJvY2Vzc2luZyBkYXRhc2V0Li4uIikKICAgIGNvbnRleHRzLCBxdWVzdGlvbnMsIG1ldGFkYXRhID0gcHJvY2Vzc19ob3Rwb3RxYV9kYXRhc2V0KAogICAgICAgIGRhdGEsIAogICAgICAgIGluY2x1ZGVfYWxsX2RvY3M9SU5DTFVERV9BTExfRE9DUywKICAgICAgICBtYXhfaXRlbXM9TUFYX0lURU1TCiAgICApCiAgICAKICAgICMgR2VuZXJhdGUgb3V0cHV0CiAgICBwcmludCgiXG5bMy8zXSBHZW5lcmF0aW5nIG91dHB1dCBmaWxlLi4uIikKICAgIGdlbmVyYXRlX291dHB1dF9maWxlKAogICAgICAgIGNvbnRleHRzLCAKICAgICAgICBxdWVzdGlvbnMsIAogICAgICAgIG1ldGFkYXRhLAogICAgICAgIG91dHB1dF9wYXRoLAogICAgICAgIGluY2x1ZGVfbWV0YWRhdGE9SU5DTFVERV9NRVRBREFUQQogICAgKQogICAgCiAgICAjIFN1bW1hcnkKICAgIHByaW50KCJcbiIgKyAiPSIgKiA2MCkKICAgIHByaW50KCJTVU1NQVJZIikKICAgIHByaW50KCI9IiAqIDYwKQogICAgcHJpbnQoZiIgIC0gVW5pcXVlIGNvbnRleHQgZ3JvdXBzOiB7bGVuKGNvbnRleHRzKX0iKQogICAgcHJpbnQoZiIgIC0gVG90YWwgUUEgcGFpcnM6IHtzdW0obGVuKHFhKSBmb3IgcWEgaW4gcXVlc3Rpb25zLnZhbHVlcygpKX0iKQogICAgcHJpbnQoZiIgIC0gT3V0cHV0IGZpbGU6IHtvdXRwdXRfcGF0aH0iKQogICAgCiAgICAjIFNob3cgcXVlc3Rpb24gdHlwZSBkaXN0cmlidXRpb24KICAgIGlmIG1ldGFkYXRhOgogICAgICAgIGFsbF90eXBlcyA9IFtdCiAgICAgICAgYWxsX2xldmVscyA9IFtdCiAgICAgICAgZm9yIG1ldGEgaW4gbWV0YWRhdGEudmFsdWVzKCk6CiAgICAgICAgICAgIGFsbF90eXBlcy5leHRlbmQobWV0YVsndHlwZSddKQogICAgICAgICAgICBhbGxfbGV2ZWxzLmV4dGVuZChtZXRhWydsZXZlbCddKQogICAgICAgIAogICAgICAgIGZyb20gY29sbGVjdGlvbnMgaW1wb3J0IENvdW50ZXIKICAgICAgICB0eXBlX2NvdW50cyA9IENvdW50ZXIoYWxsX3R5cGVzKQogICAgICAgIGxldmVsX2NvdW50cyA9IENvdW50ZXIoYWxsX2xldmVscykKICAgICAgICAKICAgICAgICBwcmludChmIlxuICBRdWVzdGlvbiBUeXBlczoiKQogICAgICAgIGZvciBxdHlwZSwgY291bnQgaW4gdHlwZV9jb3VudHMuaXRlbXMoKToKICAgICAgICAgICAgcHJpbnQoZiIgICAgLSB7cXR5cGV9OiB7Y291bnR9IikKICAgICAgICAKICAgICAgICBwcmludChmIlxuICBEaWZmaWN1bHR5IExldmVsczoiKQogICAgICAgIGZvciBsZXZlbCwgY291bnQgaW4gbGV2ZWxfY291bnRzLml0ZW1zKCk6CiAgICAgICAgICAgIHByaW50KGYiICAgIC0ge2xldmVsfToge2NvdW50fSIpCiAgICAKICAgIHByaW50KCI9IiAqIDYwKQoKCmlmIF9fbmFtZV9fID09ICJfX21haW5fXyI6CiAgICBtYWluKCkK'
    os.makedirs('data', exist_ok=True)
    with open('data/process_hotpotqa_data.py', 'wb') as f:
        f.write(base64.b64decode(script_b64))
    
    print('⚙️ Processing HotpotQA data...')
    !python data/process_hotpotqa_data.py
    
    if os.path.exists('data/processed_hotpotqa.py'):
        size_mb = os.path.getsize('data/processed_hotpotqa.py') / (1024 * 1024)
        print(f'✅ HotpotQA dataset generated ({size_mb:.1f} MB)')
    else:
        print('❌ Failed to generate HotpotQA dataset')


## 6. Run Baseline Comparison (HotpotQA)

This cell runs comparison on the HotpotQA dataset (multi-hop QA).

**Parameters:**
- `--hotpotqa`: Enable HotpotQA mode
- `--titles N`: Number of titles to evaluate (default: 10)
- `--max-questions N`: Max questions per title (default: 5)
- `--epochs N`: Digestion epochs (default: 50)
- `--verbose`: Show detailed output

In [ ]:
# Run baseline comparison on HotpotQA dataset (multi-hop QA)

!python projects/hybrid_titans/compare_baselines.py --hotpotqa --titles 50 --max-questions 5 --epochs 300 --verbose --save-results


## 7. Custom Comparison (Optional)

Run the comparison with your own parameters.

In [ ]:
# Custom parameters for sample essays
# !python projects/hybrid_titans/compare_baselines.py --essay ai --epochs 100 --topk 3

# Custom parameters for SQuAD (single-doc)
# !python projects/hybrid_titans/compare_baselines.py --squad --titles 20 --max-questions 10 --epochs 50 --verbose

# Custom parameters for multi-doc
# !python projects/hybrid_titans/compare_baselines.py --multi-doc --group-size 10 --max-questions 5 --epochs 50 --verbose

# Custom parameters for HotpotQA
# !python projects/hybrid_titans/compare_baselines.py --hotpotqa --titles 20 --max-questions 10 --epochs 300 --verbose

## 8. Download Evaluation Results

Download saved JSON results from the `evaluations/` directory to your local machine.

In [ ]:
import os, glob, shutil

eval_files = sorted(glob.glob('evaluations/*.json'))
if not eval_files:
    print('No evaluation results found in evaluations/')
else:
    print(f'Found {len(eval_files)} result file(s):')
    for f in eval_files:
        print(f'  - {f}')
    try:
        from google.colab import files
        shutil.make_archive('evaluation_results', 'zip', '.', 'evaluations')
        files.download('evaluation_results.zip')
        print('Downloading evaluation_results.zip ...')
    except ImportError:
        print('Not on Colab. Files at:', os.path.abspath('evaluations/'))